# Bản sửa ARIMA và hiệu chỉnh L12
Giữ full_5level. Chạy lại từ đầu hoặc thay phần định nghĩa baseline và chạy lại ARIMA cùng các cell so sánh.

**ARIMA_ROBUST:** thử order gốc (200 rồi ít nhất 500 vòng), ARIMA(1,0,0), (0,1,1), (0,1,0). Chỉ chấp nhận kết quả hội tụ/hữu hạn. Nếu tất cả thất bại, dùng weekday mean từ lịch sử trước từng origin và ghi `used_fallback=True`, `fallback_method`, `retry_errors` trong fit_status.csv. Đây là pipeline dự phòng, không tuyên bố tất cả chuỗi dùng ARIMA(1,1,1). Dữ liệu đầu vào sai vẫn báo lỗi. Không thể bảo đảm mọi lỗi môi trường/hết RAM đều được xử lý.

**TFT_L12_SHRINK:** thêm biến thể calibration co về BU để giảm hiệu chỉnh quá mạnh. Không cần huấn luyện lại TFT nếu các biến dự báo và evaluator còn trong kernel. Các kết quả cũ đã được xóa khỏi file để tránh nhầm với kết quả sau sửa. Bảng chọn alpha/k và factors được lưu riêng.

Tối ưu tiếp dựa trên test đã xem cần được xác nhận bằng origin mới. Các phương pháp dưới đây là thay đổi để kiểm chứng, chưa phải cải thiện M5 đã được đo.


# Hướng dẫn: TFT M5 + ARIMA, Prophet, LSTM
Notebook được bổ sung phần **12–17** ở cuối. Chạy các phần trước để tạo dữ liệu, TFT và evaluator, rồi chạy baseline.

**Đã cấu hình sẵn:** `PROFILE=full_5level`, `SMOKE=0`, toàn bộ 30.490 chuỗi và evaluation split.
Chạy lần lượt các cell; ARIMA/Prophet fit riêng 30.490 chuỗi có thể rất lâu.
Các baseline luôn dùng **toàn bộ tập chuỗi đã chọn trong notebook**, không lấy một subset khác khi đánh giá.
Khi dùng subset, các tổng store/state chỉ là tổng của subset, không phải tổng M5 đầy đủ.

- Cài thêm dependency bằng cell cài đặt ở phần 12 (bỏ dấu `#` trước `%pip`).
- Mặc định ARIMA(1,1,1); Prophet có weekly/yearly seasonality và biến SNAP/sự kiện; LSTM global có toàn bộ 51 past / 11 future features giống TFTLite.
- Không tự dò siêu tham số trên test. Các cấu hình mặc định là điểm khởi đầu, chưa phải mô hình đã tối ưu.
- Mặc định các mô hình fit tham số trên train; ARIMA cập nhật state bằng validation trước test nhưng không refit. Prophet có thể refit trước test nếu bật tùy chọn, phải ghi rõ khác biệt protocol.
- LSTM dùng weighted MSE, không occurrence head, attention hay hierarchical auxiliary loss. So sánh này là so sánh pipeline, chưa cô lập tác động riêng attention.
- Kết quả lưu trong `OUT_DIR / "baselines"`; bảng chính chấm **đủ 28 ngày**, khác bảng chọn calibration 14 ngày của notebook gốc.
- MAPE chỉ trên quan sát thực tế > 0. WMAPE/MAPE lưu dạng tỷ lệ, có thêm cột phần trăm.
- Không âm thầm fallback. Mô hình bị lỗi fit sẽ bị loại khỏi bảng đa cấp hoàn chỉnh; xem `fit_status.csv`.

In [ ]:
# Cấu hình full M5, áp dụng trước cell cấu hình mô hình.
import os
os.environ["PROFILE"] = "full_5level"
os.environ["SMOKE"] = "0"
os.environ["FORECAST_SPLIT"] = "evaluation"
os.environ["BASELINE_HISTORY_DAYS"] = "0"
print("Preset: full_5level | 30,490 series | smoke off | evaluation split")


In [ ]:
%%writefile wrmsse5.py
"""Leakage-safe WRMSSE evaluation for M5 levels L1/L2/L3/L9/L12."""

from __future__ import annotations

from typing import Mapping

import numpy as np


LEVELS = ("L1", "L2", "L3", "L9", "L12")


def group_sum(values: np.ndarray, groups: np.ndarray, n_groups: int | None = None) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    groups = np.asarray(groups, dtype=np.int64)
    if values.ndim != 2:
        raise ValueError("values must have shape [series, time]")
    if groups.ndim != 1 or len(groups) != len(values):
        raise ValueError("groups must contain one code per series")
    if n_groups is None:
        n_groups = int(groups.max()) + 1
    out = np.zeros((n_groups, values.shape[1]), dtype=np.float64)
    np.add.at(out, groups, values)
    return out


def rmsse_scale(y_history: np.ndarray) -> np.ndarray:
    """Return M5 RMSSE denominators after removing pre-launch leading zeros."""
    y_history = np.asarray(y_history, dtype=np.float64)
    if y_history.ndim != 2:
        raise ValueError("y_history must have shape [series, time]")

    scales = np.full(len(y_history), np.nan, dtype=np.float64)
    for row_index, row in enumerate(y_history):
        nonzero = np.flatnonzero(row != 0)
        if not len(nonzero):
            continue
        active = row[nonzero[0]:]
        if len(active) < 2:
            continue
        scales[row_index] = np.mean(np.diff(active) ** 2)
    return scales


def build_contexts(
    groups_by_level: Mapping[str, np.ndarray],
    sales_history: np.ndarray,
    price_history: np.ndarray,
    weight_window: int = 28,
) -> dict[str, dict[str, np.ndarray | int | str]]:
    """Precompute scale and pre-forecast dollar weights for the five levels."""
    sales_history = np.asarray(sales_history)
    price_history = np.asarray(price_history)
    if sales_history.shape != price_history.shape:
        raise ValueError("sales_history and price_history must have the same shape")
    if sales_history.ndim != 2 or sales_history.shape[1] == 0:
        raise ValueError("history must have shape [series, positive_time]")
    if weight_window <= 0:
        raise ValueError("weight_window must be positive")

    start = max(0, sales_history.shape[1] - weight_window)
    recent_sales = sales_history[:, start:].astype(np.float64)
    recent_price = np.nan_to_num(price_history[:, start:], nan=0.0).astype(np.float64)
    series_revenue = np.sum(recent_sales * recent_price, axis=1)
    contexts = {}
    for level in LEVELS:
        groups = np.asarray(groups_by_level[level], dtype=np.int64)
        n_groups = int(groups.max()) + 1
        grouped_history = group_sum(sales_history, groups, n_groups)
        scale = rmsse_scale(grouped_history)
        revenue = np.bincount(groups, weights=series_revenue, minlength=n_groups).astype(np.float64)
        valid = np.isfinite(scale) & (scale > 1e-12)
        weights = np.where(valid, np.maximum(revenue, 0.0), 0.0)
        weight_sum = weights.sum()
        if weight_sum > 0:
            weights /= weight_sum
        elif valid.any():
            weights[valid] = 1.0 / valid.sum()

        contexts[level] = {
            "level": level,
            "groups": groups,
            "n_groups": n_groups,
            "scale": scale,
            "weights": weights,
            "valid": valid,
        }
    return contexts


def evaluate_level(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    context: Mapping[str, np.ndarray | int | str],
) -> dict[str, float | int | str]:
    """Evaluate one hierarchy level using a pre-forecast evaluation context."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if y_true.shape != y_pred.shape:
        raise ValueError("y_true and y_pred must have the same shape")

    groups = np.asarray(context["groups"], dtype=np.int64)
    n_groups = int(context["n_groups"])
    scale = np.asarray(context["scale"], dtype=np.float64)
    weights = np.asarray(context["weights"], dtype=np.float64)
    valid = np.asarray(context["valid"], dtype=bool)
    yt = group_sum(y_true, groups, n_groups)
    yp = group_sum(y_pred, groups, n_groups)

    mse = np.mean((yt - yp) ** 2, axis=1)
    rmsse = np.full(n_groups, np.nan, dtype=np.float64)
    rmsse[valid] = np.sqrt(mse[valid] / scale[valid])
    wrmsse = float(np.sum(weights[valid] * rmsse[valid]))

    error = np.abs(yt - yp)
    actual_by_group = np.abs(yt).sum(axis=1)
    group_ok = actual_by_group > 1e-12
    group_bias = np.full(n_groups, np.nan, dtype=np.float64)
    group_bias[group_ok] = (yp - yt).sum(axis=1)[group_ok] / actual_by_group[group_ok]
    positive = np.abs(yt) > 0

    return {
        "level": str(context["level"]),
        "n_keys": n_groups,
        "n_valid_scales": int(valid.sum()),
        "n_obs": int(yt.size),
        "WRMSSE": wrmsse,
        "RMSE": float(np.sqrt(np.mean((yt - yp) ** 2))),
        "MAE": float(error.mean()),
        "WMAPE": float(error.sum() / max(np.abs(yt).sum(), 1e-12)),
        "MAPE": float(np.mean(error[positive] / np.abs(yt[positive]))) if positive.any() else np.nan,
        "Bias": float((yp - yt).sum() / max(np.abs(yt).sum(), 1e-12)),
        "Ratio": float(yp.sum() / max(yt.sum(), 1e-12)),
        "MeanAbsGroupBias": float(np.nanmean(np.abs(group_bias))) if group_ok.any() else np.nan,
        "P90AbsGroupBias": float(np.nanpercentile(np.abs(group_bias), 90)) if group_ok.any() else np.nan,
    }


def evaluate_wrmsse5(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    contexts: Mapping[str, Mapping[str, np.ndarray | int | str]],
) -> tuple[list[dict[str, float | int | str]], float]:
    rows = [evaluate_level(y_true, y_pred, contexts[level]) for level in LEVELS]
    return rows, float(np.mean([float(row["WRMSSE"]) for row in rows]))


In [ ]:
%%writefile hurdle_features.py
"""Leakage-safe local and hierarchical features for the M5 HurdleTFT."""

from __future__ import annotations

from collections.abc import Mapping

import numpy as np


LOCAL_STATE_COLS = (
    "nonzero_rate_7",
    "nonzero_rate_28",
    "nonzero_rate_56",
    "log_zero_run",
    "has_ever_sold",
    "log_age_since_first_sale",
    "log_mean_positive_28",
    "listed_rate_28",
    "log_last_positive_units",
)

HIER_CHANNELS = (
    "log_sales",
    "log_roll_mean_7",
    "log_roll_mean_28",
    "trend_7_28",
    "listed_ratio",
)


def rolling_sum(values: np.ndarray, window: int) -> np.ndarray:
    """Inclusive causal rolling sum along time for a two-dimensional panel."""
    values = np.asarray(values, dtype=np.float32)
    if values.ndim != 2:
        raise ValueError("values must have shape [series, time]")
    if window <= 0:
        raise ValueError("window must be positive")

    cumulative = np.cumsum(values, axis=1, dtype=np.float32)
    result = cumulative.copy()
    if window < values.shape[1]:
        result[:, window:] -= cumulative[:, :-window]
    return result


def rolling_mean(values: np.ndarray, window: int) -> np.ndarray:
    result = rolling_sum(values, window)
    counts = np.minimum(np.arange(1, values.shape[1] + 1), window).astype(np.float32)
    result /= counts[None, :]
    return result


def build_local_state_features(
    sales: np.ndarray,
    price_available: np.ndarray,
) -> np.ndarray:
    """Build L12 occurrence, lifecycle and past-listing features.

    Every value at day ``t`` depends only on observations at or before ``t``.
    Float16 storage keeps the full 30,490-series panel practical; windows are
    converted back to float32 by the dataset.
    """
    sales = np.asarray(sales, dtype=np.float32)
    price_available = np.asarray(price_available, dtype=bool)
    if sales.ndim != 2 or price_available.shape != sales.shape:
        raise ValueError("sales and price_available must share shape [series, time]")
    if np.any(sales < 0):
        raise ValueError("sales must be non-negative")

    n_series, n_days = sales.shape
    output = np.empty((n_series, n_days, len(LOCAL_STATE_COLS)), dtype=np.float16)
    positive = sales > 0
    positive_float = positive.astype(np.float32)

    for name, window in (
        ("nonzero_rate_7", 7),
        ("nonzero_rate_28", 28),
        ("nonzero_rate_56", 56),
    ):
        output[:, :, LOCAL_STATE_COLS.index(name)] = rolling_mean(
            positive_float, window
        ).astype(np.float16)

    run = np.zeros(n_series, dtype=np.int16)
    first_sale = np.full(n_series, -1, dtype=np.int16)
    last_positive = np.zeros(n_series, dtype=np.float32)
    zero_run_idx = LOCAL_STATE_COLS.index("log_zero_run")
    ever_idx = LOCAL_STATE_COLS.index("has_ever_sold")
    age_idx = LOCAL_STATE_COLS.index("log_age_since_first_sale")
    last_positive_idx = LOCAL_STATE_COLS.index("log_last_positive_units")
    for day in range(n_days):
        sold = positive[:, day]
        first_sale[(first_sale < 0) & sold] = day
        run = np.where(sold, 0, np.minimum(run + 1, 364)).astype(np.int16)
        last_positive = np.where(sold, sales[:, day], last_positive)
        seen = first_sale >= 0
        age = np.where(seen, day - first_sale + 1, 0)
        output[:, day, zero_run_idx] = np.log1p(run).astype(np.float16)
        output[:, day, ever_idx] = seen.astype(np.float16)
        output[:, day, age_idx] = np.log1p(age).astype(np.float16)
        output[:, day, last_positive_idx] = np.log1p(last_positive).astype(np.float16)

    positive_count_28 = rolling_sum(positive_float, 28)
    positive_sales_28 = rolling_sum(sales, 28)
    mean_positive_28 = np.divide(
        positive_sales_28,
        positive_count_28,
        out=np.zeros_like(positive_sales_28),
        where=positive_count_28 > 0,
    )
    output[:, :, LOCAL_STATE_COLS.index("log_mean_positive_28")] = np.log1p(
        mean_positive_28
    ).astype(np.float16)
    output[:, :, LOCAL_STATE_COLS.index("listed_rate_28")] = rolling_mean(
        price_available.astype(np.float32), 28
    ).astype(np.float16)
    return output


def group_sum(values: np.ndarray, groups: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float32)
    groups = np.asarray(groups, dtype=np.int64)
    if values.ndim != 2 or groups.ndim != 1 or len(groups) != len(values):
        raise ValueError("invalid values/groups shape")
    if len(groups) == 0 or np.any(groups < 0):
        raise ValueError("groups must contain non-negative codes")
    result = np.zeros((int(groups.max()) + 1, values.shape[1]), dtype=np.float32)
    np.add.at(result, groups, values)
    return result


def build_hierarchy_context(
    sales: np.ndarray,
    price_available: np.ndarray,
    groups_by_level: Mapping[str, np.ndarray],
    levels: tuple[str, ...] = ("L1", "L2", "L3", "L9"),
) -> dict[str, np.ndarray]:
    """Build compact parent histories without broadcasting them to L12."""
    sales = np.asarray(sales, dtype=np.float32)
    price_available = np.asarray(price_available, dtype=bool)
    if sales.ndim != 2 or price_available.shape != sales.shape:
        raise ValueError("sales and price_available must share shape [series, time]")

    contexts = {}
    for level in levels:
        groups = np.asarray(groups_by_level[level], dtype=np.int64)
        aggregate = group_sum(sales, groups)
        roll_7 = rolling_mean(aggregate, 7)
        roll_28 = rolling_mean(aggregate, 28)
        trend = np.clip((roll_7 - roll_28) / (roll_28 + 1.0), -5.0, 5.0)
        listed_count = group_sum(price_available.astype(np.float32), groups)
        group_size = np.bincount(groups, minlength=aggregate.shape[0]).astype(np.float32)
        listed_ratio = listed_count / np.maximum(group_size[:, None], 1.0)
        contexts[level] = np.stack(
            [
                np.log1p(aggregate),
                np.log1p(roll_7),
                np.log1p(roll_28),
                trend,
                listed_ratio,
            ],
            axis=-1,
        ).astype(np.float32)
    return contexts


def fit_feature_scaler(values: np.ndarray, train_end: int) -> tuple[np.ndarray, np.ndarray]:
    """Fit feature-wise moments strictly before the one-based train cutoff."""
    values = np.asarray(values)
    if values.ndim < 2 or not 0 < train_end <= values.shape[1]:
        raise ValueError("invalid train_end")
    axes = tuple(range(values.ndim - 1))
    train = values[:, :train_end].astype(np.float32)
    mean = train.mean(axis=axes, dtype=np.float64).astype(np.float32)
    std = train.std(axis=axes, dtype=np.float64).astype(np.float32)
    std[std < 1e-6] = 1.0
    return mean, std


def hierarchy_feature_names(
    levels: tuple[str, ...] = ("L1", "L2", "L3", "L9"),
) -> list[str]:
    return [f"{level}_{channel}" for level in levels for channel in HIER_CHANNELS]


# TFT M5 - zero-inflation test (L1/L2/L3/L9/L12)

Notebook test doc lap. Muc tieu:
- Kiem tra pipeline TFT tren M5 voi du lieu that, tap trung vao 5 cap L1/L2/L3/L9/L12.
- Xu ly zero inflation bang occurrence head, Tweedie NLL va dac trung lifecycle/zero-aware.
- Dua context qua khu L1/L2/L3/L9 vao encoder ma khong broadcast xuong toan panel.
- So sanh bottom-up (BU), calibration theo store, global va series-level.
- Chon model tren validation, khong chon tren test.
- Chay smoke test tren Kaggle T4 hoac local CPU voi subset nho.

Luu y: WRMSSE-5 chi gom L1/L2/L3/L9/L12, khong phai WRMSSE 12 cap official.
Scale va dollar weights van tuan theo nguyen tac M5 va chi dung history truoc forecast.



In [ ]:
import os
import sys
import json
import math
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from wrmsse5 import LEVELS, build_contexts, evaluate_wrmsse5
except ModuleNotFoundError:
    from kaggle.wrmsse5 import LEVELS, build_contexts, evaluate_wrmsse5

try:
    from hurdle_features import (
        HIER_CHANNELS,
        LOCAL_STATE_COLS,
        build_hierarchy_context,
        build_local_state_features,
        fit_feature_scaler,
        hierarchy_feature_names,
    )
except ModuleNotFoundError:
    from kaggle.hurdle_features import (
        HIER_CHANNELS,
        LOCAL_STATE_COLS,
        build_hierarchy_context,
        build_local_state_features,
        fit_feature_scaler,
        hierarchy_feature_names,
    )

import matplotlib
if not Path("/kaggle").exists():
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROFILE = os.environ.get("PROFILE", "full_5level").lower()
SMOKE = os.environ.get("SMOKE", "0" if PROFILE == "full_5level" else "1") == "1"
N_SERIES = int(os.environ.get("N_SERIES", 120 if SMOKE else 3000))
EPOCHS = int(os.environ.get("EPOCHS", 2 if SMOKE else 12))
MAX_TRAIN_ORIGINS = int(os.environ.get("MAX_TRAIN_ORIGINS", 120 if SMOKE else 240))
MAX_TRAIN_ORIGINS = None if MAX_TRAIN_ORIGINS <= 0 else MAX_TRAIN_ORIGINS
BATCH = int(os.environ.get("BATCH", 128 if SMOKE else 512))
SEQ = 56
H = 28
D_MODEL = int(os.environ.get("D_MODEL", 32 if SMOKE else 64))
HEADS = int(os.environ.get("HEADS", 2 if SMOKE else 4))
DROPOUT = 0.10
LR = float(os.environ.get("LR", 1e-3))
WEIGHT_DECAY = 1e-4
TWEEDIE_P = 1.3
LOSS = os.environ.get("LOSS", "tweedie")
SERIES_WEIGHT_POWER = float(os.environ.get("SERIES_WEIGHT_POWER", "0.5"))
SERIES_CAL_K = float(os.environ.get("SERIES_CAL_K", "5.0"))
HIERARCHICAL_LOSS_WEIGHT = float(os.environ.get("HIERARCHICAL_LOSS_WEIGHT", "0.2"))
ZERO_LOSS_WEIGHT = float(os.environ.get("ZERO_LOSS_WEIGHT", "0.1"))
CALIBRATION_DAYS = int(os.environ.get("CALIBRATION_DAYS", "14"))
BEST_METRIC = os.environ.get("BEST_METRIC", "wrmsse5").lower()
FORECAST_SPLIT = (
    "evaluation"
    if PROFILE == "full_5level"
    else os.environ.get("FORECAST_SPLIT", "validation").lower()
)
if PROFILE == "5000_l12":
    SMOKE = False
    N_SERIES = int(os.environ.get("N_SERIES", "5000"))
    EPOCHS = int(os.environ.get("EPOCHS", "10"))
    MAX_TRAIN_ORIGINS = int(os.environ.get("MAX_TRAIN_ORIGINS", "160"))
    BATCH = int(os.environ.get("BATCH", "256"))
    D_MODEL = int(os.environ.get("D_MODEL", "64"))
    HEADS = int(os.environ.get("HEADS", "4"))
    SERIES_WEIGHT_POWER = float(os.environ.get("SERIES_WEIGHT_POWER", "0.5"))
    BEST_METRIC = os.environ.get("BEST_METRIC", "wrmsse5").lower()
elif PROFILE == "full_5level":
    SMOKE = False
    N_SERIES = 30490
    EPOCHS = int(os.environ.get("FULL_EPOCHS", "20"))
    MAX_TRAIN_ORIGINS = int(os.environ.get("MAX_TRAIN_ORIGINS", "160"))
    BATCH = int(os.environ.get("BATCH", "512"))
    D_MODEL = int(os.environ.get("D_MODEL", "64"))
    HEADS = int(os.environ.get("HEADS", "4"))
    BEST_METRIC = os.environ.get("BEST_METRIC", "wrmsse5").lower()
assert BEST_METRIC in {"l12", "wrmsse5"}
assert FORECAST_SPLIT in {"validation", "evaluation"}
assert 0.0 <= HIERARCHICAL_LOSS_WEIGHT <= 1.0
assert ZERO_LOSS_WEIGHT >= 0.0
assert 0 < CALIBRATION_DAYS < H
SEED = 42

if FORECAST_SPLIT == "evaluation":
    TRAIN_START, TRAIN_END = 57, 1885
    VAL_START, VAL_END = 1886, 1913
    TEST_START, TEST_END = 1914, 1941
    SALES_FILE = "sales_train_evaluation.csv"
else:
    TRAIN_START, TRAIN_END = 57, 1857
    VAL_START, VAL_END = 1858, 1885
    TEST_START, TEST_END = 1886, 1913
    SALES_FILE = "sales_train_validation.csv"
VAL_ORIGIN = VAL_START - 2
TEST_ORIGIN = TEST_START - 2
TRAIN_LAST_ORIGIN = TRAIN_END - H - 1

assert H == 28
assert TRAIN_LAST_ORIGIN >= SEQ - 1


def safe_str(value):
    return str(value).encode("ascii", "backslashreplace").decode("ascii")


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = 0 if os.name == "nt" else (2 if Path("/kaggle").exists() else 0)

if os.environ.get("OUT_DIR"):
    OUT_DIR = Path(os.environ["OUT_DIR"])
elif Path("/kaggle/working").exists():
    OUT_DIR = Path("/kaggle/working")
else:
    OUT_DIR = Path("test_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE_DESC = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print("device:", DEVICE, "|", DEVICE_DESC)
print("smoke:", SMOKE, "| series:", N_SERIES, "| epochs:", EPOCHS)
print("max_train_origins:", MAX_TRAIN_ORIGINS, "| batch:", BATCH)
print(
    "profile:", PROFILE or "custom",
    "| series_weight_power:", SERIES_WEIGHT_POWER,
    "| series_cal_k:", SERIES_CAL_K,
    "| best_metric:", BEST_METRIC,
    "| hierarchical_loss_weight:", HIERARCHICAL_LOSS_WEIGHT,
    "| zero_loss_weight:", ZERO_LOSS_WEIGHT,
    "| calibration_days:", CALIBRATION_DAYS,
    "| forecast_split:", FORECAST_SPLIT,
)
print("output:", safe_str(OUT_DIR.resolve()))



Mac dinh notebook chay full 30.490 series, evaluation split, toi da 20 epoch.
Chay nhanh: dat `PROFILE=smoke SMOKE=1` (120 series, 2 epoch).
Chay 5.000 series tap trung L12 tren Kaggle T4: dat `PROFILE=5000_l12`
(them `%env PROFILE=5000_l12` truoc cell config, hoac sua dong `PROFILE = ...`).
Tuong duong: `SMOKE=0 N_SERIES=5000 EPOCHS=10 BATCH=256 D_MODEL=64 HEADS=4 MAX_TRAIN_ORIGINS=160 SERIES_WEIGHT_POWER=0.5 BEST_METRIC=l12`
Doi so epoch full bang `FULL_EPOCHS`; profile full luon khoa N_SERIES=30.490.
Co the doi `LOSS=mse` de so sanh voi Tweedie, hoac `MAX_TRAIN_ORIGINS=0` de dung toan bo origin.
Can dat `wrmsse5.py` cung thu muc khi chay source tren Kaggle.



## 1. Load M5

Notebook tim data theo thu tu: `M5_DATA_ROOT`, Kaggle input, `../data/raw`, `data/raw`.
Local hien co `sales_train_validation.csv` (d_1-d_1913), nen test dung d_1886-d_1913.
Luu y: ban local trong repo co the chi la sample 100 series (2 stores, 1 dept),
vi vay smoke local chi de kiem tra pipeline; Kaggle full moi co 30.490 series.
Tren Kaggle co the doi sang `sales_train_evaluation.csv` va test d_1914-d_1941.



In [ ]:
def find_data_root():
    candidates = []
    if os.environ.get("M5_DATA_ROOT"):
        configured = Path(os.environ["M5_DATA_ROOT"]).expanduser()
        candidates.append(configured.parent if configured.is_file() else configured)
    candidates += [
        Path("/kaggle/input/m5-forecasting-accuracy"),
        Path("../data/raw"),
        Path("data/raw"),
        Path("../../data/raw"),
    ]
    for path in candidates:
        if (path / SALES_FILE).exists():
            return path.resolve()

    # Kaggle dataset slugs and uploaded dataset layouts can add extra folders.
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob(SALES_FILE))
        complete = [
            match.parent
            for match in matches
            if (match.parent / "calendar.csv").exists()
            and (match.parent / "sell_prices.csv").exists()
        ]
        if complete:
            return complete[0].resolve()

        attached = sorted(path.name for path in kaggle_input.iterdir())
        attached_text = ", ".join(attached) if attached else "(khong co dataset nao)"
    else:
        attached_text = "/kaggle/input khong ton tai"
    raise FileNotFoundError(
        f"Khong tim thay {SALES_FILE} kem calendar.csv va sell_prices.csv. "
        "Tren Kaggle, chon Add Input > M5 Forecasting Accuracy, sau do Run All lai. "
        f"Hoac dat M5_DATA_ROOT toi thu muc chua ba file. Input hien co: {attached_text}"
    )


DATA_ROOT = find_data_root()
print("data_root:", safe_str(DATA_ROOT))


def select_series(sales, n_series, seed=42):
    if n_series >= len(sales):
        return sales.reset_index(drop=True)

    stores = sorted(sales["store_id"].unique())
    per_store = max(1, int(math.ceil(n_series / len(stores))))
    parts = []

    for store in stores:
        g = sales[sales["store_id"].eq(store)].copy()
        first = g.sort_values(["dept_id", "item_id"]).groupby("dept_id", sort=True).head(1)
        rest = g.drop(first.index).sample(frac=1.0, random_state=seed)
        picked = pd.concat([first, rest]).drop_duplicates("id").head(per_store)
        parts.append(picked)

    out = pd.concat(parts).drop_duplicates("id")
    if len(out) < n_series:
        rest = sales[~sales["id"].isin(out["id"])]
        if len(rest):
            out = pd.concat([out, rest.sample(n=min(n_series - len(out), len(rest)), random_state=seed)])
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


sales_all = pd.read_csv(DATA_ROOT / SALES_FILE, dtype={"id": str})
meta_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = sorted([c for c in sales_all.columns if c.startswith("d_")], key=lambda x: int(x[2:]))
n_days = len(day_cols)

assert n_days >= TEST_END, f"Can it nhat {TEST_END} ngay, dang co {n_days}"
if PROFILE == "full_5level" and len(sales_all) != 30490:
    raise ValueError(
        f"PROFILE=full_5level can dung 30.490 series, nhung {SALES_FILE} chi co "
        f"{len(sales_all):,}. Kiem tra lai M5_DATA_ROOT va dataset Kaggle."
    )
sales_sel = select_series(sales_all, N_SERIES, SEED)
series_ids = sales_sel["id"].tolist()
n_series = len(series_ids)
if PROFILE == "full_5level" and n_series != 30490:
    raise ValueError(
        f"Full profile phai chon 30.490 series, hien chi chon {n_series:,}."
    )

print("series selected:", n_series)
print("days:", n_days)
print(sales_sel.groupby(["store_id", "state_id"]).size().to_string())



## 2. Build matrices: sales, causal past price, calendar



In [ ]:
SALES_DF = sales_sel.set_index("id").loc[series_ids, day_cols]
SALES_DF.columns = np.arange(1, n_days + 1)
SALES_DF = SALES_DF.astype(np.float32).fillna(0.0)
SALES = SALES_DF.to_numpy(dtype=np.float32)

cal = pd.read_csv(DATA_ROOT / "calendar.csv")
cal["d_num"] = cal["d"].str[2:].astype(np.int16)
cal["event_name_any"] = cal["event_name_1"].fillna(cal["event_name_2"])
cal["event_type"] = cal["event_type_1"].fillna(cal["event_type_2"]).fillna("None")
cal["has_event"] = cal["event_name_any"].notna().astype(np.float32)
cal["event_type_code"] = cal["event_type"].astype("category").cat.codes.astype(np.float32) + 1.0
cal = cal.sort_values("d_num").set_index("d_num")
cal = cal.reindex(np.arange(1, n_days + 1)).reset_index()
cal["wday"] = cal["wday"].fillna(0).astype(np.float32)
cal["month"] = cal["month"].fillna(0).astype(np.float32)
cal["year"] = cal["year"].fillna(0).astype(np.float32)
cal["wm_yr_wk"] = cal["wm_yr_wk"].fillna(0).astype(np.int64)
cal["snap_CA"] = cal["snap_CA"].fillna(0).astype(np.float32)
cal["snap_TX"] = cal["snap_TX"].fillna(0).astype(np.float32)
cal["snap_WI"] = cal["snap_WI"].fillna(0).astype(np.float32)
cal["has_event"] = cal["has_event"].fillna(0).astype(np.float32)
cal["event_type_code"] = cal["event_type_code"].fillna(0).astype(np.float32)

CAL = {
    "is_weekend": cal["wday"].isin([1, 2]).to_numpy(np.float32),
    "sin_wday": np.sin(2.0 * np.pi * cal["wday"].to_numpy(np.float32) / 7.0).astype(np.float32),
    "cos_wday": np.cos(2.0 * np.pi * cal["wday"].to_numpy(np.float32) / 7.0).astype(np.float32),
    "sin_month": np.sin(2.0 * np.pi * cal["month"].to_numpy(np.float32) / 12.0).astype(np.float32),
    "cos_month": np.cos(2.0 * np.pi * cal["month"].to_numpy(np.float32) / 12.0).astype(np.float32),
    "sin_year": np.sin(2.0 * np.pi * cal["d_num"].to_numpy(np.float32) / 365.25).astype(np.float32),
    "cos_year": np.cos(2.0 * np.pi * cal["d_num"].to_numpy(np.float32) / 365.25).astype(np.float32),
    "has_event": cal["has_event"].to_numpy(np.float32),
    "event_type_code": cal["event_type_code"].to_numpy(np.float32),
}

SNAP = np.zeros((n_series, n_days), dtype=np.float32)
for state, col in [("CA", "snap_CA"), ("TX", "snap_TX"), ("WI", "snap_WI")]:
    mask = sales_sel["state_id"].eq(state).to_numpy()
    if mask.any():
        SNAP[mask] = cal[col].to_numpy(np.float32)[None, :]

prices = pd.read_csv(
    DATA_ROOT / "sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
)
pairs = sales_sel[["store_id", "item_id"]].drop_duplicates()
prices = prices.merge(pairs, on=["store_id", "item_id"], how="inner")

# Expand weekly prices directly to days. Avoiding a 59M-row melt/merge keeps the
# full 30,490-series profile within Kaggle RAM.
price_weekly = prices.pivot_table(
    index=["store_id", "item_id"],
    columns="wm_yr_wk",
    values="sell_price",
    aggfunc="last",
)
pair_order = sales_sel.set_index("id").loc[series_ids, ["store_id", "item_id"]]
pair_index = pd.MultiIndex.from_frame(pair_order)
price_weekly = price_weekly.reindex(pair_index)
week_positions = pd.Index(price_weekly.columns).get_indexer(cal["wm_yr_wk"].to_numpy())
price_values = np.full((n_series, n_days), np.nan, dtype=np.float32)
valid_week = week_positions >= 0
price_values[:, valid_week] = price_weekly.to_numpy(np.float32)[:, week_positions[valid_week]]
PRICE_RAW = pd.DataFrame(price_values, index=series_ids, columns=np.arange(1, n_days + 1))
PRICE_OBS = PRICE_RAW.to_numpy(np.float32)
PRICE_AVAILABLE = np.isfinite(PRICE_OBS)

# Price is unknown in the forecast horizon. Past prices are filled causally;
# pre-listing days stay at zero and are identified by is_listed.
PRICE_DF = PRICE_RAW.ffill(axis=1).fillna(0.0)
PRICE = PRICE_DF.to_numpy(np.float32)

price_median_28 = (
    PRICE_DF.shift(1, axis=1).T.rolling(28, min_periods=1).median().T
)
PRICE_REL = (PRICE_DF / price_median_28.replace(0.0, np.nan))
PRICE_REL = PRICE_REL.replace([np.inf, -np.inf], np.nan).fillna(1.0).clip(0.0, 5.0)
PRICE_REL = PRICE_REL.to_numpy(np.float32)
PROMO = (PRICE_REL < 0.95).astype(np.float32)

print("sales matrix:", SALES.shape)
print("price matrix:", PRICE.shape)
print("past price availability rate:", float(PRICE_AVAILABLE.mean()))
print("zero rate:", float((SALES == 0).mean()))
print("promo rate:", float(PROMO.mean()))

del prices, pairs, price_weekly, price_values, pair_order, pair_index, cal, sales_all



## 3. Feature engineering

Past features duoc tinh tai forecast origin. Future chi gom calendar/SNAP/event biet truoc.
Gia va promotion tuong lai khong duoc dua vao decoder.
Cac lag 1/7/14/28/35/56/364 deu an toan voi direct multi-horizon vi chi lay tu qua khu.



In [ ]:
PAST_COLS = [
    "sales", "lag_1", "lag_7", "lag_14", "lag_28", "lag_35", "lag_56", "lag_364",
    "roll_mean_7", "roll_mean_28", "roll_mean_56", "roll_std_28",
    "sell_price", "price_rel", "promo", "is_listed", "snap", "has_event",
    "sin_wday", "cos_wday", "sin_month", "cos_month",
]
FUT_COLS = [
    "is_weekend", "sin_wday", "cos_wday", "sin_month", "cos_month",
    "sin_year", "cos_year", "snap", "has_event", "event_type_code",
]
FEATURE_COLS = list(dict.fromkeys(PAST_COLS + FUT_COLS))

FEAT = np.zeros((n_series, n_days, len(FEATURE_COLS)), dtype=np.float32)


def store_feature(name, values):
    FEAT[:, :, FEATURE_COLS.index(name)] = values


# Write one feature at a time instead of retaining ~9 GB of intermediate maps.
S = SALES_DF
store_feature("sales", SALES)
for k in [1, 7, 14, 28, 35, 56, 364]:
    store_feature(f"lag_{k}", S.shift(k, axis=1).fillna(0.0).to_numpy(np.float32))

for window in [7, 28, 56]:
    store_feature(
        f"roll_mean_{window}",
        S.T.rolling(window, min_periods=1).mean().T.to_numpy(np.float32),
    )
store_feature(
    "roll_std_28",
    S.T.rolling(28, min_periods=2).std().T.fillna(0.0).to_numpy(np.float32),
)

store_feature("sell_price", PRICE)
store_feature("price_rel", PRICE_REL)
store_feature("promo", PROMO)
store_feature("is_listed", PRICE_AVAILABLE.astype(np.float32))
store_feature("snap", SNAP)
for name in [
    "is_weekend", "has_event", "event_type_code", "sin_wday", "cos_wday",
    "sin_month", "cos_month", "sin_year", "cos_year",
]:
    store_feature(name, CAL[name][None, :])

PAST_IDX = [FEATURE_COLS.index(c) for c in PAST_COLS]
FUT_IDX = [FEATURE_COLS.index(c) for c in FUT_COLS]

STATIC_COLS = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
STATIC_DF = sales_sel.set_index("id").loc[series_ids, STATIC_COLS]
STATIC = np.zeros((n_series, len(STATIC_COLS)), dtype=np.int64)
STATIC_CARD = []
for j, col in enumerate(STATIC_COLS):
    codes, uniques = pd.factorize(STATIC_DF[col].astype(str), sort=True)
    STATIC[:, j] = codes + 1
    STATIC_CARD.append(len(uniques) + 1)


def dense_codes(codes):
    uniques = np.unique(codes)
    mapping = {value: i for i, value in enumerate(uniques)}
    return np.array([mapping[v] for v in codes], dtype=np.int64)


store_code = STATIC[:, STATIC_COLS.index("store_id")]
dept_code = STATIC[:, STATIC_COLS.index("dept_id")]
state_code = STATIC[:, STATIC_COLS.index("state_id")]
GROUPS = {
    "L1": np.zeros(n_series, dtype=np.int64),
    "L2": dense_codes(state_code),
    "L3": dense_codes(store_code),
    "L9": dense_codes(store_code * 1000 + dept_code),
    "L12": np.arange(n_series, dtype=np.int64),
}
if PROFILE == "full_5level":
    actual_group_counts = {level: int(GROUPS[level].max()) + 1 for level in LEVELS}
    expected_group_counts = {"L1": 1, "L2": 3, "L3": 10, "L9": 70, "L12": 30490}
    if actual_group_counts != expected_group_counts:
        raise ValueError(
            f"Hierarchy full khong dung: {actual_group_counts}; can {expected_group_counts}"
        )
HIERARCHY_GROUP_TENSORS = {
    level: torch.as_tensor(GROUPS[level], dtype=torch.long, device=DEVICE)
    for level in ("L1", "L2", "L3", "L9")
}

HIER_LEVELS = ("L1", "L2", "L3", "L9")
LOCAL_STATE = build_local_state_features(SALES, PRICE_AVAILABLE)
HIER_CONTEXT = build_hierarchy_context(SALES, PRICE_AVAILABLE, GROUPS, HIER_LEVELS)
HIER_CONTEXT_COLS = hierarchy_feature_names(HIER_LEVELS)
MODEL_PAST_COLS = PAST_COLS + list(LOCAL_STATE_COLS) + HIER_CONTEXT_COLS

VAL_EVAL_CONTEXTS = build_contexts(
    GROUPS,
    SALES[:, :VAL_START - 1],
    PRICE_OBS[:, :VAL_START - 1],
)
TEST_EVAL_CONTEXTS = build_contexts(
    GROUPS,
    SALES[:, :TEST_START - 1],
    PRICE_OBS[:, :TEST_START - 1],
)
CAL_SELECT_EVAL_CONTEXTS = build_contexts(
    GROUPS,
    SALES[:, :VAL_START - 1 + CALIBRATION_DAYS],
    PRICE_OBS[:, :VAL_START - 1 + CALIBRATION_DAYS],
)

del S
del SALES_DF, PRICE_DF, PRICE_RAW, price_median_28

print(
    "features:", FEAT.shape,
    "| past base/local/hierarchy:", len(PAST_COLS), len(LOCAL_STATE_COLS), len(HIER_CONTEXT_COLS),
    "| future:", len(FUT_COLS),
)
print("static cards:", STATIC_CARD)
print({k: int(v.max()) + 1 for k, v in GROUPS.items()})



## 4. Split, scaler and window dataset



In [ ]:
train_rev_28 = (
    SALES[:, TRAIN_END - 28:TRAIN_END].astype(np.float64)
    * np.nan_to_num(PRICE_OBS[:, TRAIN_END - 28:TRAIN_END], nan=0.0).astype(np.float64)
).sum(axis=1)
train_rev = train_rev_28
if SERIES_WEIGHT_POWER > 0 and train_rev.sum() > 0:
    series_weight = np.power(np.maximum(train_rev, 0.0), SERIES_WEIGHT_POWER)
    series_weight = series_weight / max(float(series_weight.mean()), 1e-9)
    series_weight = np.clip(series_weight, 0.25, 4.0)
else:
    series_weight = np.ones(n_series, dtype=np.float64)
series_weight = series_weight.astype(np.float32)
del PRICE, PRICE_REL, PROMO, SNAP, PRICE_AVAILABLE, PRICE_OBS
print(
    "series weight:",
    f"min={series_weight.min():.3f}",
    f"median={np.median(series_weight):.3f}",
    f"max={series_weight.max():.3f}",
)

train_feat = FEAT[:, :TRAIN_END, :]
FEAT_MEAN = train_feat.mean(axis=(0, 1)).astype(np.float32)
FEAT_STD = train_feat.std(axis=(0, 1)).astype(np.float32)
FEAT_STD[FEAT_STD < 1e-6] = 1.0
FEAT -= FEAT_MEAN[None, None, :]
FEAT /= FEAT_STD[None, None, :]
del train_feat

LOCAL_STATE_MEAN, LOCAL_STATE_STD = fit_feature_scaler(LOCAL_STATE, TRAIN_END)
HIER_CONTEXT_MEAN = {}
HIER_CONTEXT_STD = {}
for level in HIER_LEVELS:
    mean, std = fit_feature_scaler(HIER_CONTEXT[level], TRAIN_END)
    HIER_CONTEXT_MEAN[level] = mean
    HIER_CONTEXT_STD[level] = std
    HIER_CONTEXT[level] -= mean[None, None, :]
    HIER_CONTEXT[level] /= std[None, None, :]

first_origin = SEQ - 1
train_origin_values = list(range(first_origin, TRAIN_LAST_ORIGIN + 1))
if MAX_TRAIN_ORIGINS is not None and len(train_origin_values) > MAX_TRAIN_ORIGINS:
    pick = np.linspace(0, len(train_origin_values) - 1, MAX_TRAIN_ORIGINS).round().astype(int)
    train_origin_values = [train_origin_values[i] for i in pick]


class SeriesOriginGrid:
    """Indexable Cartesian product without materializing millions of tuples."""
    def __init__(self, n_series, origins):
        self.n_series = n_series
        self.origins = origins

    def __len__(self):
        return self.n_series * len(self.origins)

    def __getitem__(self, index):
        series, origin_pos = divmod(index, len(self.origins))
        return series, self.origins[origin_pos]


train_origins = SeriesOriginGrid(n_series, train_origin_values)

val_origins = [(s, VAL_ORIGIN) for s in range(n_series)]
test_origins = [(s, TEST_ORIGIN) for s in range(n_series)]

print("train windows:", len(train_origins))
print("val windows:", len(val_origins))
print("test windows:", len(test_origins))
print("train target range:", "d_%d-d_%d" % (TRAIN_START, TRAIN_END))
print("val target range:", "d_%d-d_%d" % (VAL_START, VAL_END))
print("test target range:", "d_%d-d_%d" % (TEST_START, TEST_END))


class M5WindowDataset(Dataset):
    def __init__(
        self,
        feat,
        local_state,
        hierarchy_context,
        groups,
        target,
        static,
        origins,
        past_idx,
        fut_idx,
        sample_weight=None,
        seq=56,
        horizon=28,
    ):
        self.feat = feat
        self.local_state = local_state
        self.hierarchy_context = hierarchy_context
        self.groups = groups
        self.target = target
        self.static = static
        self.origins = origins
        self.past_idx = np.asarray(past_idx, dtype=np.int64)
        self.fut_idx = np.asarray(fut_idx, dtype=np.int64)
        self.sample_weight = sample_weight
        self.seq = seq
        self.horizon = horizon

    def __len__(self):
        return len(self.origins)

    def __getitem__(self, i):
        s, origin = self.origins[i]
        start = origin - self.seq + 1
        past_base = self.feat[s, start:origin + 1][:, self.past_idx]
        past_local = self.local_state[s, start:origin + 1].astype(np.float32)
        past_local = (past_local - LOCAL_STATE_MEAN) / LOCAL_STATE_STD
        past_hierarchy = np.concatenate([
            self.hierarchy_context[level][self.groups[level][s], start:origin + 1]
            for level in HIER_LEVELS
        ], axis=1)
        past = np.concatenate([past_base, past_local, past_hierarchy], axis=1)
        future = self.feat[s, origin + 1:origin + 1 + self.horizon][:, self.fut_idx]
        target = self.target[s, origin + 1:origin + 1 + self.horizon]
        weight = 1.0 if self.sample_weight is None else float(self.sample_weight[s])
        mask = np.full(self.horizon, weight, dtype=np.float32)
        return (
            torch.from_numpy(np.ascontiguousarray(past)),
            torch.from_numpy(np.ascontiguousarray(future)),
            torch.from_numpy(self.static[s].astype(np.int64)),
            torch.from_numpy(np.ascontiguousarray(target)),
            torch.from_numpy(np.ascontiguousarray(mask)),
            torch.tensor(s, dtype=torch.long),
            torch.tensor(origin, dtype=torch.long),
        )


class OriginBatchSampler:
    """Build batches from one origin so aggregate losses compare the same days."""
    def __init__(self, n_series, n_origins, batch_size, seed=42):
        self.n_series = n_series
        self.n_origins = n_origins
        self.batch_size = batch_size
        self.seed = seed
        self.epoch = 0

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        for origin_pos in rng.permutation(self.n_origins):
            series_order = rng.permutation(self.n_series)
            for start in range(0, self.n_series, self.batch_size):
                series_batch = series_order[start:start + self.batch_size]
                yield [int(s) * self.n_origins + int(origin_pos) for s in series_batch]

    def __len__(self):
        return self.n_origins * int(math.ceil(self.n_series / self.batch_size))


dataset_args = (FEAT, LOCAL_STATE, HIER_CONTEXT, GROUPS, SALES, STATIC)
train_ds = M5WindowDataset(
    *dataset_args,
    train_origins,
    PAST_IDX,
    FUT_IDX,
    sample_weight=series_weight,
    seq=SEQ,
    horizon=H,
)
val_ds = M5WindowDataset(
    *dataset_args, val_origins, PAST_IDX, FUT_IDX, seq=SEQ, horizon=H
)
test_ds = M5WindowDataset(
    *dataset_args, test_origins, PAST_IDX, FUT_IDX, seq=SEQ, horizon=H
)

if HIERARCHICAL_LOSS_WEIGHT > 0:
    train_loader = DataLoader(
        train_ds,
        batch_sampler=OriginBatchSampler(n_series, len(train_origin_values), BATCH, SEED),
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
    )
else:
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        drop_last=False,
    )
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
)

batch = next(iter(train_loader))
print("past:", tuple(batch[0].shape), "| future:", tuple(batch[1].shape))
print("static:", tuple(batch[2].shape), "| target:", tuple(batch[3].shape), "| mask:", tuple(batch[4].shape))



## 5. TFT-lite model

Model gom static embeddings, variable selection networks, LSTM encoder/decoder,
interpretable-style attention va mean head. Head dung softplus de dam bao mu > 0.



In [ ]:
class GRN(nn.Module):
    def __init__(self, d_model, ctx_dim=0, dropout=0.1):
        super().__init__()
        in_dim = d_model + ctx_dim
        self.fc1 = nn.Linear(in_dim, 2 * d_model)
        self.fc2 = nn.Linear(2 * d_model, d_model)
        self.gate = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, ctx=None):
        h = torch.cat([x, ctx], dim=-1) if ctx is not None else x
        h = F.elu(self.fc1(h))
        h = self.drop(h)
        h = self.fc2(h)
        h = self.drop(h)
        gate = torch.sigmoid(self.gate(h))
        return self.norm(x + gate * h)


class VSN(nn.Module):
    def __init__(self, n_features, d_model, dropout=0.1):
        super().__init__()
        self.weight_net = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ELU(),
            nn.Linear(d_model, n_features),
        )
        self.proj = nn.Linear(n_features, d_model)
        self.grn = GRN(d_model, dropout=dropout)

    def forward(self, x, ctx):
        w = torch.softmax(self.weight_net(ctx), dim=-1)
        return self.grn(self.proj(x * w.unsqueeze(1)))


class StaticEmbedding(nn.Module):
    def __init__(self, cards, d_model):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(card, d_model) for card in cards])
        self.proj = nn.Linear(d_model * len(cards), d_model)

    def forward(self, static):
        parts = [emb(static[:, i]) for i, emb in enumerate(self.embeddings)]
        return self.proj(torch.cat(parts, dim=-1))


class TFTLite(nn.Module):
    def __init__(self, n_past, n_future, cards, d_model=32, heads=2, dropout=0.1, horizon=28):
        super().__init__()
        self.horizon = horizon
        self.static = StaticEmbedding(cards, d_model)
        self.past_vsn = VSN(n_past, d_model, dropout)
        self.fut_vsn = VSN(n_future, d_model, dropout)
        self.encoder = nn.LSTM(d_model, d_model, batch_first=True)
        self.decoder = nn.LSTM(d_model, d_model, batch_first=True)
        self.static_ctx = nn.Linear(d_model, d_model)
        self.enrich = GRN(d_model, ctx_dim=d_model, dropout=dropout)
        self.attn = nn.MultiheadAttention(d_model, heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.ff = GRN(d_model, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, 1)
        self.zero_head = nn.Linear(d_model, 1)

    def forward(self, past, future, static, return_zero_logits=False):
        ctx = self.static(static)
        past = self.past_vsn(past, ctx)
        future = self.fut_vsn(future, ctx)
        enc, (h, c) = self.encoder(past)
        dec, _ = self.decoder(future, (h, c))
        x = torch.cat([enc, dec], dim=1)
        x = x + self.static_ctx(ctx).unsqueeze(1)
        x = self.enrich(x, ctx.unsqueeze(1).expand(-1, x.size(1), -1))

        length = x.size(1)
        causal = torch.triu(
            torch.ones(length, length, device=x.device, dtype=torch.bool),
            diagonal=1,
        )
        attn, _ = self.attn(x, x, x, attn_mask=causal, need_weights=False)
        x = self.attn_norm(x + self.drop(attn))
        x = self.ff(x)
        forecast_state = x[:, -self.horizon:, :]
        severity = F.softplus(self.head(forecast_state).squeeze(-1)) + 1e-4
        zero_logits = self.zero_head(forecast_state).squeeze(-1)
        mean = torch.sigmoid(zero_logits) * severity
        if return_zero_logits:
            return mean, zero_logits
        return mean


model = TFTLite(
    n_past=len(MODEL_PAST_COLS),
    n_future=len(FUT_COLS),
    cards=STATIC_CARD,
    d_model=D_MODEL,
    heads=HEADS,
    dropout=DROPOUT,
    horizon=H,
).to(DEVICE)
print(model)
print("parameters:", sum(p.numel() for p in model.parameters()))



## 6. Loss, metrics and trainer



In [ ]:
def tweedie_loss_elementwise(y, mu, p=TWEEDIE_P):
    mu = mu.clamp_min(1e-6)
    return -y * torch.pow(mu, 1.0 - p) / (1.0 - p) + torch.pow(mu, 2.0 - p) / (2.0 - p)


def masked_loss(y, mu, mask):
    if LOSS == "tweedie":
        loss = tweedie_loss_elementwise(y, mu)
    else:
        loss = (mu - y) ** 2
    return (loss * mask).sum() / mask.sum().clamp_min(1.0)


def zero_occurrence_loss(y, logits, mask):
    occurrence = (y > 0).to(logits.dtype)
    loss = F.binary_cross_entropy_with_logits(logits, occurrence, reduction="none")
    return (loss * mask).sum() / mask.sum().clamp_min(1.0)


def hierarchical_aux_loss(y, mu, mask, series_index):
    """Relative aggregate error for L1/L2/L3/L9 within an origin-aligned batch."""
    # Keep index_add inputs in one dtype under CUDA autocast.
    valid = (mask > 0).float()
    pred = mu.float() * valid
    actual = y.float() * valid
    losses = []

    for level in ("L1", "L2", "L3", "L9"):
        codes = HIERARCHY_GROUP_TENSORS[level][series_index]
        _, inverse = torch.unique(codes, sorted=False, return_inverse=True)
        n_groups = int(inverse.max().item()) + 1
        pred_sum = torch.zeros(n_groups, pred.size(1), device=pred.device, dtype=pred.dtype)
        actual_sum = torch.zeros_like(pred_sum)
        count = torch.zeros_like(pred_sum)
        pred_sum.index_add_(0, inverse, pred)
        actual_sum.index_add_(0, inverse, actual)
        count.index_add_(0, inverse, valid)

        group_valid = count > 0
        denominator = actual_sum.detach().abs() + count.clamp_min(1.0)
        relative_error = (pred_sum - actual_sum) / denominator
        losses.append((relative_error[group_valid] ** 2).mean())

    return torch.stack(losses).mean()


def eval_all(y_true, y_pred, contexts, tag):
    rows, overall = evaluate_wrmsse5(y_true, y_pred, contexts)
    df = pd.DataFrame(rows)
    df.insert(0, "model", tag)
    return df, overall


def make_summary_row(frame, tag, overall):
    l1 = frame[frame["level"].eq("L1")].iloc[0]
    l12 = frame[frame["level"].eq("L12")].iloc[0]
    row = {
        "model": tag,
        "WRMSSE_5levels": overall,
        "L1_WMAPE": l1["WMAPE"],
        "L12_WMAPE": l12["WMAPE"],
        "L12_Bias": l12["Bias"],
        "L12_Ratio": l12["Ratio"],
    }
    for level in LEVELS:
        row[f"{level}_WRMSSE"] = frame.loc[frame["level"].eq(level), "WRMSSE"].iloc[0]
    return row


def seasonal_naive(sales, target_start, lag, horizon=28):
    history_end = target_start - 1
    if history_end < lag:
        raise ValueError(f"Need at least {lag} history days")
    pattern = sales[:, history_end - lag:history_end]
    repeats = int(math.ceil(horizon / lag))
    return np.tile(pattern, (1, repeats))[:, :horizon].astype(np.float32)


def weekday_mean_naive(sales, target_start, horizon=28, weeks=4):
    history_end = target_start - 1
    window = 7 * weeks
    if history_end < window:
        raise ValueError(f"Need at least {window} history days")
    recent = sales[:, history_end - window:history_end].reshape(len(sales), weeks, 7)
    weekday_mean = recent.mean(axis=1)
    return np.tile(weekday_mean, (1, int(math.ceil(horizon / 7))))[:, :horizon].astype(np.float32)


def zero_metrics(y_true, y_pred, threshold=0.5):
    actual = y_true.ravel()
    pred = y_pred.ravel()
    actual_zero = actual < 0.5
    pred_zero = pred < threshold
    tp = float((actual_zero & pred_zero).sum())
    fp = float((~actual_zero & pred_zero).sum())
    fn = float((actual_zero & ~pred_zero).sum())
    precision = tp / max(tp + fp, 1.0)
    recall = tp / max(tp + fn, 1.0)
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-9)

    nonzero = ~actual_zero
    return {
        "zero_precision": precision,
        "zero_recall": recall,
        "zero_f1": f1,
        "zero_rate_actual": float(actual_zero.mean()),
        "zero_rate_pred": float(pred_zero.mean()),
        "mae_nonzero": float(np.abs(actual[nonzero] - pred[nonzero]).mean()) if nonzero.any() else np.nan,
        "bias_nonzero": float((pred[nonzero] - actual[nonzero]).mean()) if nonzero.any() else np.nan,
        "bias_zero": float((pred[actual_zero] - actual[actual_zero]).mean()) if actual_zero.any() else np.nan,
    }


def fit_store_factors(pred, true, groups, low=0.8, high=1.2):
    factors = {}
    for g in np.unique(groups):
        mask = groups == g
        denominator = float(pred[mask].sum())
        factor = 1.0 if denominator <= 1e-9 else float(true[mask].sum() / denominator)
        factors[g] = float(np.clip(factor, low, high))
    return factors


def apply_factors(pred, groups, factors):
    out = pred.copy()
    for g, factor in factors.items():
        out[groups == g] *= factor
    return out


def fit_series_factors(pred, true, k=5.0, low=0.5, high=2.0):
    pred_sum = pred.sum(axis=1).astype(np.float64)
    true_sum = true.sum(axis=1).astype(np.float64)
    factors = (true_sum + k) / np.maximum(pred_sum + k, 1e-9)
    factors = np.where(pred_sum + true_sum > 0, factors, 1.0)
    return np.clip(factors, low, high).astype(np.float32)


def apply_series_factors(pred, factors):
    return pred * factors[:, None]


def run_epoch(model, loader, optimizer=None, scaler=None, scheduler=None):
    train = optimizer is not None
    model.train(train)
    total_loss = 0.0
    total_base_loss = 0.0
    total_hier_loss = 0.0
    total_zero_loss = 0.0
    total_rows = 0

    for past, future, static, target, mask, series_index, _ in loader:
        past = past.to(DEVICE, non_blocking=True)
        future = future.to(DEVICE, non_blocking=True)
        static = static.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        series_index = series_index.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                mu, zero_logits = model(past, future, static, return_zero_logits=True)
                base_loss = masked_loss(target, mu, mask)
                zero_loss = zero_occurrence_loss(target, zero_logits, mask)
                if train and HIERARCHICAL_LOSS_WEIGHT > 0:
                    hier_loss = hierarchical_aux_loss(target, mu, mask, series_index)
                else:
                    hier_loss = torch.zeros((), device=target.device, dtype=base_loss.dtype)
                loss = (
                    base_loss
                    + HIERARCHICAL_LOSS_WEIGHT * hier_loss
                    + ZERO_LOSS_WEIGHT * zero_loss
                )

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                if scheduler is not None:
                    scheduler.step()

        total_loss += float(loss.item()) * target.size(0)
        total_base_loss += float(base_loss.item()) * target.size(0)
        total_hier_loss += float(hier_loss.item()) * target.size(0)
        total_zero_loss += float(zero_loss.item()) * target.size(0)
        total_rows += target.size(0)

    denominator = max(total_rows, 1)
    return {
        "loss": total_loss / denominator,
        "base_loss": total_base_loss / denominator,
        "hier_loss": total_hier_loss / denominator,
        "zero_loss": total_zero_loss / denominator,
    }


@torch.no_grad()
def predict_ds(model, dataset, batch_size=BATCH):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
    )
    model.eval()
    preds, targets, masks = [], [], []
    for past, future, static, target, mask, _, _ in loader:
        past = past.to(DEVICE, non_blocking=True)
        future = future.to(DEVICE, non_blocking=True)
        static = static.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            mu = model(past, future, static)
        preds.append(mu.float().cpu().numpy())
        targets.append(target.numpy())
        masks.append(mask.numpy())
    return (
        np.concatenate(preds, axis=0),
        np.concatenate(targets, axis=0),
        np.concatenate(masks, axis=0),
    )



## 7. Train smoke/full



In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = max(1, EPOCHS * len(train_loader))
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    total_steps=total_steps,
    pct_start=0.3,
    anneal_strategy="cos",
)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

best_score = float("inf")
best_state = None
patience = 0
history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_stats = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scaler=scaler,
        scheduler=scheduler,
    )
    val_pred, val_true, val_valid = predict_ds(model, val_ds)
    val_pred = val_pred * val_valid
    val_df, val_wrmsse = eval_all(val_true, val_pred, VAL_EVAL_CONTEXTS, "val_bu")
    val_l1_wmape = float(val_df.loc[val_df["level"].eq("L1"), "WMAPE"].iloc[0])
    val_l12_wrmsse = float(val_df.loc[val_df["level"].eq("L12"), "WRMSSE"].iloc[0])
    score = val_l12_wrmsse if BEST_METRIC == "l12" else val_wrmsse

    history.append({
        "epoch": epoch,
        "train_loss": train_stats["loss"],
        "train_base_loss": train_stats["base_loss"],
        "train_hier_loss": train_stats["hier_loss"],
        "train_zero_loss": train_stats["zero_loss"],
        "learning_rate": optimizer.param_groups[0]["lr"],
        "val_wrmsse_5levels": val_wrmsse,
        "val_l12_wrmsse": val_l12_wrmsse,
        "val_l1_wmape": val_l1_wmape,
        "val_score": score,
        "seconds": time.time() - start,
    })

    improved = score < best_score - 1e-4
    if improved:
        best_score = score
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        patience = 0
    else:
        patience += 1

    print(
        f"epoch {epoch:02d} | loss {train_stats['loss']:.5f} "
        f"(base {train_stats['base_loss']:.5f}, hier {train_stats['hier_loss']:.5f}) | "
        f"zero {train_stats['zero_loss']:.5f} | "
        f"lr {optimizer.param_groups[0]['lr']:.2e} | "
        f"val_wrmsse5 {val_wrmsse:.5f} | val_l12 {val_l12_wrmsse:.5f} | "
        f"val_l1_wmape {val_l1_wmape:.5f} | {history[-1]['seconds']:.1f}s"
    )

    if patience >= 3:
        print("early stopping")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"loaded best checkpoint | {BEST_METRIC}:", best_score)

history_df = pd.DataFrame(history)
display(history_df) if "display" in globals() else print(history_df.to_string(index=False))



## 8. Predict test, calibration and 5-level metrics



In [ ]:
val_pred, val_true, val_valid = predict_ds(model, val_ds)
test_pred, test_true, test_valid = predict_ds(model, test_ds)
val_pred = val_pred * val_valid
test_pred = test_pred * test_valid

baseline_variants = []
for baseline_tag, builder in [
    ("NAIVE_7", lambda start: seasonal_naive(SALES, start, 7, H)),
    ("NAIVE_28", lambda start: seasonal_naive(SALES, start, 28, H)),
    ("WEEKDAY_MEAN_4", lambda start: weekday_mean_naive(SALES, start, H, 4)),
]:
    val_baseline = builder(VAL_START) * val_valid
    test_baseline = builder(TEST_START) * test_valid
    baseline_variants.append((
        baseline_tag,
        val_baseline[:, CALIBRATION_DAYS:],
        test_baseline,
    ))

store_groups = GROUPS["L3"]
cal_pred = val_pred[:, :CALIBRATION_DAYS]
cal_true = val_true[:, :CALIBRATION_DAYS]
select_pred = val_pred[:, CALIBRATION_DAYS:]
select_true = val_true[:, CALIBRATION_DAYS:]

# Select the calibration strategy on days not used to fit its factors.
selection_store_factors = fit_store_factors(cal_pred, cal_true, store_groups, low=0.8, high=1.2)
selection_global_factor = float(np.clip(cal_true.sum() / max(cal_pred.sum(), 1e-9), 0.8, 1.2))
selection_store = apply_factors(select_pred, store_groups, selection_store_factors)
selection_series_factors = fit_series_factors(
    apply_factors(cal_pred, store_groups, selection_store_factors),
    cal_true,
    k=SERIES_CAL_K,
    low=0.5,
    high=2.0,
)
selection_variants = {
    "BU": select_pred,
    "STORE_CAL": selection_store,
    "GLOBAL_CAL": select_pred * selection_global_factor,
    "SERIES_CAL": apply_series_factors(selection_store, selection_series_factors),
}

# Once the strategy is selected, refit factors on all validation days for test.
store_factors = fit_store_factors(val_pred, val_true, store_groups, low=0.8, high=1.2)
global_factor = float(np.clip(val_true.sum() / max(val_pred.sum(), 1e-9), 0.8, 1.2))
val_store_full = apply_factors(val_pred, store_groups, store_factors)
series_factors = fit_series_factors(val_store_full, val_true, k=SERIES_CAL_K, low=0.5, high=2.0)
test_variants = {
    "BU": test_pred,
    "STORE_CAL": apply_factors(test_pred, store_groups, store_factors),
    "GLOBAL_CAL": test_pred * global_factor,
    "SERIES_CAL": apply_series_factors(
        apply_factors(test_pred, store_groups, store_factors),
        series_factors,
    ),
}
test_bu = test_variants["BU"]
test_store = test_variants["STORE_CAL"]
test_global = test_variants["GLOBAL_CAL"]
test_series = test_variants["SERIES_CAL"]

print("refit global calibration factor:", round(global_factor, 4))
print("refit store factors:", {k: round(v, 4) for k, v in store_factors.items()})
print(
    "series factors:",
    f"min={series_factors.min():.3f}",
    f"median={np.median(series_factors):.3f}",
    f"max={series_factors.max():.3f}",
)

variants = [
    (tag, selection_variants[tag], test_variants[tag])
    for tag in ["BU", "STORE_CAL", "GLOBAL_CAL", "SERIES_CAL"]
]

val_metric_frames = []
val_summary_rows = []
test_metric_frames = []
test_summary_rows = []
for tag, vp, tp in variants:
    val_frame, val_overall = eval_all(select_true, vp, CAL_SELECT_EVAL_CONTEXTS, tag)
    val_metric_frames.append(val_frame)
    val_summary_rows.append(make_summary_row(val_frame, tag, val_overall))
    test_frame, test_overall = eval_all(test_true, tp, TEST_EVAL_CONTEXTS, tag)
    test_metric_frames.append(test_frame)
    test_summary_rows.append(make_summary_row(test_frame, tag, test_overall))

val_metrics_df = pd.concat(val_metric_frames, ignore_index=True)
val_summary_df = pd.DataFrame(val_summary_rows)
if BEST_METRIC == "l12":
    val_summary_df = val_summary_df.sort_values(["L12_WRMSSE", "WRMSSE_5levels"]).reset_index(drop=True)
else:
    val_summary_df = val_summary_df.sort_values(["WRMSSE_5levels", "L12_WRMSSE"]).reset_index(drop=True)
best_tag = str(val_summary_df.iloc[0]["model"])
val_summary_df["selected"] = val_summary_df["model"].eq(best_tag)

metrics_df = pd.concat(test_metric_frames, ignore_index=True)
summary_df = pd.DataFrame(test_summary_rows)
summary_df["selected"] = summary_df["model"].eq(best_tag)
summary_df = summary_df.sort_values(
    ["selected", "WRMSSE_5levels"],
    ascending=[False, True],
).reset_index(drop=True)

baseline_val_frames = []
baseline_test_frames = []
baseline_summary_rows = []
for tag, vp, tp in baseline_variants:
    val_frame, val_overall = eval_all(select_true, vp, CAL_SELECT_EVAL_CONTEXTS, tag)
    test_frame, test_overall = eval_all(test_true, tp, TEST_EVAL_CONTEXTS, tag)
    baseline_val_frames.append(val_frame)
    baseline_test_frames.append(test_frame)
    baseline_summary_rows.append(make_summary_row(test_frame, tag, test_overall))
baseline_val_metrics_df = pd.concat(baseline_val_frames, ignore_index=True)
baseline_metrics_df = pd.concat(baseline_test_frames, ignore_index=True)
baseline_summary_df = pd.DataFrame(baseline_summary_rows).sort_values("WRMSSE_5levels")

best_baseline_by_level = baseline_val_metrics_df.groupby("level")["WRMSSE"].min()
selected_val = val_metrics_df[val_metrics_df["model"].eq(best_tag)].set_index("level")
beats_baseline = all(
    selected_val.loc[level, "WRMSSE"] <= best_baseline_by_level.loc[level]
    for level in LEVELS
)
val_summary_df["beats_baseline_all_levels"] = val_summary_df["model"].eq(best_tag) & beats_baseline
summary_df["beats_baseline_all_levels_validation"] = summary_df["model"].eq(best_tag) & beats_baseline

print("validation summary:")
print(val_summary_df.to_string(index=False))
print("test summary:")
print(summary_df.to_string(index=False))
print("seasonal baseline test summary:")
print(baseline_summary_df.to_string(index=False))

pred_map = test_variants
best_pred = pred_map[best_tag]
best_metrics = metrics_df[metrics_df["model"].eq(best_tag)].reset_index(drop=True)
print("chosen by validation:", best_tag)
print(best_metrics.to_string(index=False))

zero_df = pd.DataFrame([
    {"model": tag, **zero_metrics(test_true, tp)}
    for tag, _, tp in variants
])
print(zero_df.to_string(index=False))

zero_threshold_rows = []
for tag, _, tp in variants:
    for threshold in [0.10, 0.25, 0.50, 0.75, 1.00]:
        zero_threshold_rows.append({
            "model": tag,
            "threshold": threshold,
            **zero_metrics(test_true, tp, threshold=threshold),
        })
zero_threshold_df = pd.DataFrame(zero_threshold_rows)
print(zero_threshold_df.to_string(index=False))

history_df.to_csv(OUT_DIR / "history_test.csv", index=False)
metrics_df.to_csv(OUT_DIR / "metrics_test.csv", index=False)
summary_df.to_csv(OUT_DIR / "summary_test.csv", index=False)
val_metrics_df.to_csv(OUT_DIR / "val_metrics_test.csv", index=False)
val_summary_df.to_csv(OUT_DIR / "val_summary_test.csv", index=False)
baseline_val_metrics_df.to_csv(OUT_DIR / "baseline_val_metrics_test.csv", index=False)
baseline_metrics_df.to_csv(OUT_DIR / "baseline_metrics_test.csv", index=False)
baseline_summary_df.to_csv(OUT_DIR / "baseline_summary_test.csv", index=False)
zero_df.to_csv(OUT_DIR / "zero_metrics_test.csv", index=False)
zero_threshold_df.to_csv(OUT_DIR / "zero_threshold_test.csv", index=False)
np.savez_compressed(
    OUT_DIR / "predictions_test.npz",
    ids=np.array(series_ids),
    y_true=test_true,
    y_pred_bu=test_bu,
    y_pred_store_cal=test_store,
    y_pred_global_cal=test_global,
    y_pred_series_cal=test_series,
    series_factors=series_factors,
    valid_mask=test_valid,
)
torch.save(
    {
        "state_dict": model.state_dict(),
        "past_cols": MODEL_PAST_COLS,
        "past_base_cols": PAST_COLS,
        "local_state_cols": list(LOCAL_STATE_COLS),
        "hierarchy_context_cols": HIER_CONTEXT_COLS,
        "future_cols": FUT_COLS,
        "static_card": STATIC_CARD,
        "config": {
            "seq": SEQ,
            "horizon": H,
            "d_model": D_MODEL,
            "heads": HEADS,
            "dropout": DROPOUT,
            "tweedie_p": TWEEDIE_P,
            "loss": LOSS,
            "series_weight_power": SERIES_WEIGHT_POWER,
            "series_cal_k": SERIES_CAL_K,
            "hierarchical_loss_weight": HIERARCHICAL_LOSS_WEIGHT,
            "zero_loss_weight": ZERO_LOSS_WEIGHT,
            "best_metric": BEST_METRIC,
            "forecast_split": FORECAST_SPLIT,
            "future_price_known": False,
            "feature_version": "v2_local_state_hierarchy_context",
            "evaluator": "wrmsse5_v1",
            "seed": SEED,
        },
        "feat_mean": FEAT_MEAN,
        "feat_std": FEAT_STD,
        "local_state_mean": LOCAL_STATE_MEAN,
        "local_state_std": LOCAL_STATE_STD,
        "hierarchy_context_mean": HIER_CONTEXT_MEAN,
        "hierarchy_context_std": HIER_CONTEXT_STD,
    },
    OUT_DIR / "model_test.pt",
)
with open(OUT_DIR / "config_test.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "smoke": SMOKE,
            "profile": PROFILE or "custom",
            "n_series": n_series,
            "epochs": EPOCHS,
            "max_train_origins": MAX_TRAIN_ORIGINS,
            "batch": BATCH,
            "seq": SEQ,
            "horizon": H,
            "d_model": D_MODEL,
            "heads": HEADS,
            "loss": LOSS,
            "tweedie_p": TWEEDIE_P,
            "series_weight_power": SERIES_WEIGHT_POWER,
            "series_cal_k": SERIES_CAL_K,
            "hierarchical_loss_weight": HIERARCHICAL_LOSS_WEIGHT,
            "zero_loss_weight": ZERO_LOSS_WEIGHT,
            "calibration_days": CALIBRATION_DAYS,
            "best_metric": BEST_METRIC,
            "forecast_split": FORECAST_SPLIT,
            "future_price_known": False,
            "feature_version": "v2_local_state_hierarchy_context",
            "sales_file": SALES_FILE,
            "evaluator": "wrmsse5_v1",
            "best_model": best_tag,
            "beats_baseline_all_levels_validation": bool(beats_baseline),
            "best_val_l12_wrmsse": float(
                val_summary_df.loc[val_summary_df["model"].eq(best_tag), "L12_WRMSSE"].iloc[0]
            ),
            "best_test_wrmsse_5levels": float(
                summary_df.loc[summary_df["model"].eq(best_tag), "WRMSSE_5levels"].iloc[0]
            ),
        },
        f,
        indent=2,
    )
print("saved to:", OUT_DIR.resolve())



## 9. Zero-rate buckets and error analysis



In [ ]:
zero_rate_train = (SALES[:, :TRAIN_END] == 0).mean(axis=1)
buckets = pd.cut(
    zero_rate_train,
    bins=[-0.01, 0.25, 0.50, 0.75, 0.99, 1.0],
    labels=["<25%", "25-50%", "50-75%", "75-99%", "99-100%"],
)

bucket_rows = []
for label in buckets.categories:
    mask = (buckets == label)
    if not mask.any():
        continue
    yt = test_true[mask]
    yp = best_pred[mask]
    bucket_rows.append({
        "zero_bucket": label,
        "n_series": int(mask.sum()),
        "actual_sum": float(yt.sum()),
        "pred_sum": float(yp.sum()),
        "MAE": float(np.abs(yt - yp).mean()),
        "WMAPE": float(np.abs(yt - yp).sum() / max(np.abs(yt).sum(), 1e-9)),
        "Bias": float((yp - yt).sum() / max(np.abs(yt).sum(), 1e-9)),
        "Ratio": float(yp.sum() / max(yt.sum(), 1e-9)),
    })

bucket_df = pd.DataFrame(bucket_rows)
print(bucket_df.to_string(index=False))
bucket_df.to_csv(OUT_DIR / "zero_bucket_test.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(zero_rate_train, bins=20, color="#4c72b0", alpha=0.85)
axes[0].set_title("Training zero rate by series")
axes[0].set_xlabel("zero rate")
axes[0].set_ylabel("series")

daily_true = test_true.sum(axis=0)
daily_bu = test_bu.sum(axis=0)
daily_best = best_pred.sum(axis=0)
axes[1].plot(daily_true, label="actual", linewidth=2)
axes[1].plot(daily_bu, label="BU", linestyle="--")
axes[1].plot(daily_best, label=best_tag, linestyle=":")
axes[1].set_title("Test total units by day")
axes[1].set_xlabel("day in horizon")
axes[1].set_ylabel("units")
axes[1].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "test_diagnostics.png", dpi=120)
plt.show()



## 10. Feature importance from VSN



In [ ]:
past_weight_sum = torch.zeros(len(MODEL_PAST_COLS), device=DEVICE)
future_weight_sum = torch.zeros(len(FUT_COLS), device=DEVICE)
weight_rows = 0
with torch.no_grad():
    for _, _, static, _, _, _, _ in val_loader:
        static = static.to(DEVICE, non_blocking=True)
        ctx = model.static(static)
        past_weight_sum += torch.softmax(model.past_vsn.weight_net(ctx), dim=-1).sum(dim=0)
        future_weight_sum += torch.softmax(model.fut_vsn.weight_net(ctx), dim=-1).sum(dim=0)
        weight_rows += static.size(0)
past_weights = (past_weight_sum / max(weight_rows, 1)).cpu().numpy()
future_weights = (future_weight_sum / max(weight_rows, 1)).cpu().numpy()

importance_df = pd.concat([
    pd.DataFrame({"feature": MODEL_PAST_COLS, "weight": past_weights, "kind": "past"}),
    pd.DataFrame({"feature": FUT_COLS, "weight": future_weights, "kind": "future"}),
], ignore_index=True).sort_values("weight", ascending=False)

print(importance_df.to_string(index=False))
importance_df.to_csv(OUT_DIR / "feature_importance_test.csv", index=False)



## 11. Doc ket qua

- `metrics_test.csv`: metric theo tung cap cho BU, store calibration, global calibration, series calibration.
- `summary_test.csv`: so sanh 4 bien the; cot `selected` duoc chon tren validation, khong phai test.
- `val_metrics_test.csv`, `val_summary_test.csv`: metric validation de chon model.
- `history_test.csv`: base/hierarchical loss, learning rate va val metric theo epoch.
- `zero_metrics_test.csv`, `zero_threshold_test.csv`, `zero_bucket_test.csv`: chat luong du doan zero.
- `predictions_test.npz`: du doan test de audit, gom `y_pred_series_cal`.
- `model_test.pt`, `config_test.json`: checkpoint va cau hinh.

Buoc tiep theo:
1. Chay full 5 cap: `PROFILE=full_5level FORECAST_SPLIT=evaluation`.
2. Chon checkpoint theo WRMSSE-5, khong toi uu rieng L12.
3. Ablate local lifecycle va hierarchy context de do dong gop theo tung cap.


## 12. Baseline configuration and dependencies
Cấu hình bằng biến môi trường trước khi chạy cell dưới. `BASELINE_HISTORY_DAYS=0` dùng toàn bộ lịch sử train;
giá trị dương giới hạn lịch sử của ARIMA/Prophet và được ghi trong manifest.
`BASELINE_N_JOBS=1` mặc định tránh sao chép panel lớn sang nhiều process.
`BASELINE_REFIT_TEST=0` giữ tham số thống kê fit trên train; đặt 1 để refit bằng train+validation (không có test).
Dùng cùng protocol khi công bố so sánh. LSTM/TFT không tự refit trên validation.

Nguồn API: [statsmodels ARIMA append](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMAResults.append.html),
[Prophet regressors](https://facebook.github.io/prophet/docs/seasonality%2C_holiday_effects%2C_and_regressors.html).

In [ ]:
# Bỏ dấu # và chạy nếu môi trường chưa có thư viện:
# %pip install statsmodels prophet joblib
import importlib.util
import importlib.metadata
import logging
from joblib import Parallel, delayed

required_baseline_vars = ["SALES", "FEAT", "FEATURE_COLS", "train_ds", "val_ds", "test_ds",
                          "VAL_EVAL_CONTEXTS", "TEST_EVAL_CONTEXTS", "test_variants", "best_tag"]
missing_vars = [x for x in required_baseline_vars if x not in globals()]
if missing_vars:
    raise RuntimeError("Chạy các phần 1–11 trước: thiếu " + ", ".join(missing_vars))
BASELINE_MODELS = tuple(x.strip().upper() for x in os.getenv("BASELINE_MODELS", "ARIMA,PROPHET,LSTM").split(",") if x.strip())
if not set(BASELINE_MODELS) <= {"ARIMA", "PROPHET", "LSTM"}:
    raise ValueError("BASELINE_MODELS chỉ nhận ARIMA,PROPHET,LSTM")
for tag, pkg in [("ARIMA", "statsmodels"), ("PROPHET", "prophet")]:
    if tag in BASELINE_MODELS and importlib.util.find_spec(pkg) is None:
        raise ImportError(f"Thiếu {pkg}; chạy %pip install statsmodels prophet joblib")
BASELINE_HISTORY_DAYS = int(os.getenv("BASELINE_HISTORY_DAYS", "0"))
BASELINE_N_JOBS = int(os.getenv("BASELINE_N_JOBS", "1"))
BASELINE_REFIT_TEST = os.getenv("BASELINE_REFIT_TEST", "0") == "1"
ARIMA_ORDER = tuple(int(v) for v in os.getenv("ARIMA_ORDER", "1,1,1").split(","))
ARIMA_MAXITER = int(os.getenv("ARIMA_MAXITER", "200"))
LSTM_EPOCHS = int(os.getenv("LSTM_EPOCHS", "2" if SMOKE else "20"))
LSTM_HIDDEN = int(os.getenv("LSTM_HIDDEN", str(D_MODEL)))
LSTM_PATIENCE = int(os.getenv("LSTM_PATIENCE", "3"))
LSTM_LR = float(os.getenv("LSTM_LR", "0.001"))
assert len(ARIMA_ORDER) == 3 and min(ARIMA_ORDER) >= 0
assert BASELINE_HISTORY_DAYS >= 0 and BASELINE_N_JOBS != 0
assert LSTM_EPOCHS > 0 and LSTM_PATIENCE > 0 and LSTM_HIDDEN > 0
assert TRAIN_END == VAL_START - 1 and VAL_END == TEST_START - 1
BASELINE_DIR = OUT_DIR / "baselines"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
BASELINE_VAL_TRUE = SALES[:, VAL_START-1:VAL_END].copy()
BASELINE_TEST_TRUE = SALES[:, TEST_START-1:TEST_END].copy()
BASELINE_VAL_PRED, BASELINE_TEST_PRED = {}, {}
BASELINE_STATUS = []
print("Baseline models:", BASELINE_MODELS, "| series:", n_series,
      "| statistical refit before test:", BASELINE_REFIT_TEST)

## 13. ARIMA and Prophet: local models, bottom-up forecasts
ARIMA là mô hình đơn biến, không sử dụng ngoại sinh. Prophet dùng SNAP và cờ sự kiện biết trước;
không sử dụng giá tương lai. LSTM/TFT có bộ đầu vào phong phú hơn: đây là khác biệt cần ghi trong báo cáo.

Các chuỗi hoàn toàn hằng được xử lý bằng dự báo hằng và ghi `constant_history`.
Nếu ARIMA không hội tụ hoặc phát sinh lỗi, không dùng dự báo đó để tính điểm hoàn chỉnh.
Có thể tăng `ARIMA_MAXITER`, điều chỉnh order trên validation rồi chạy lại.

In [ ]:
# Đọc ngày thật từ calendar; không hard-code ngày bắt đầu M5.
_bcal = pd.read_csv(DATA_ROOT / "calendar.csv")
_bcal["d_num"] = _bcal["d"].str[2:].astype(int)
BASELINE_DATES = pd.to_datetime(_bcal.set_index("d_num").loc[np.arange(1, n_days+1), "date"]).to_numpy()
del _bcal
# Khôi phục các regressor từ FEAT đã chuẩn hóa bằng scaler TRAIN.
PROPHET_REGRESSORS = ("snap", "has_event")
_reg_idx = [FEATURE_COLS.index(x) for x in PROPHET_REGRESSORS]
BASELINE_REG = (FEAT[:, :, _reg_idx] * FEAT_STD[_reg_idx] + FEAT_MEAN[_reg_idx]).astype(np.float32)
BASELINE_REG = np.rint(BASELINE_REG)  # Hai biến này là binary, loại sai số chuẩn hóa.

def baseline_positive_prediction(values, horizon):
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    if values.shape != (horizon,) or not np.isfinite(values).all():
        raise ValueError("Forecast không đúng shape hoặc có NaN/Inf")
    return np.maximum(values, 0).astype(np.float32)

def _fit_local_baseline_once(tag, y, dates, regressors, train_end, val_end, test_end,
                       history_days=0, refit_test=False, order=(1,1,1), maxiter=200):
    """Never read test targets: caller passes y only through val_end.
    Returns val/test forecasts, plus auditable fit status.
    """
    from time import perf_counter
    started = perf_counter()
    hv, ht = val_end-train_end, test_end-val_end
    y = np.asarray(y, dtype=np.float64)
    if len(y) != val_end:
        raise ValueError("y phải kết thúc ở validation cutoff, không chứa test")
    details = []
    def predict_from_cutoff(cutoff, steps):
        start = max(0, cutoff-history_days) if history_days else 0
        history = y[start:cutoff]
        if not len(history) or not np.isfinite(history).all():
            raise ValueError("History rỗng hoặc không hữu hạn")
        if np.ptp(history) < 1e-12:
            details.append("constant_history")
            return np.full(steps, history[-1], dtype=np.float64), None
        if tag == "ARIMA":
            from statsmodels.tsa.arima.model import ARIMA
            with warnings.catch_warnings(record=True) as captured:
                warnings.simplefilter("always")
                fitted = ARIMA(history, order=order).fit(method_kwargs={"maxiter": maxiter})
            details.extend(str(w.message)[:250] for w in captured)
            if not fitted.mle_retvals.get("converged", True):
                raise RuntimeError("ARIMA optimizer did not converge")
            return fitted.forecast(steps), fitted
        if tag != "PROPHET":
            raise ValueError(tag)
        from prophet import Prophet
        frame = pd.DataFrame({"ds": dates[start:cutoff], "y": history})
        future = pd.DataFrame({"ds": dates[cutoff:cutoff+steps]})
        fitted = Prophet(weekly_seasonality=True, yearly_seasonality=len(history)>=730,
                         daily_seasonality=False, uncertainty_samples=0,
                         changepoint_prior_scale=0.05)
        for j, name in enumerate(PROPHET_REGRESSORS):
            # Drop constant TRAIN regressors; do not inspect future to select variables.
            if np.ptp(regressors[start:cutoff, j]) > 1e-8:
                fitted.add_regressor(name, standardize=False)
                frame[name] = regressors[start:cutoff, j]
                future[name] = regressors[cutoff:cutoff+steps, j]
        fitted.fit(frame)
        return fitted.predict(future)["yhat"].to_numpy(), fitted
    try:
        # Prophet without refit forecasts the whole validation+test calendar in one fit.
        steps = hv+ht if tag == "PROPHET" and not refit_test else hv
        first, fitted = predict_from_cutoff(train_end, steps)
        vp = first[:hv]
        if refit_test:
            tp, _ = predict_from_cutoff(val_end, ht)
        elif tag == "PROPHET":
            tp = first[hv:hv+ht]
        elif fitted is None:
            tp = np.full(ht, y[train_end-1])
            details.append("frozen_constant_train_model")
        else:
            # Assimilate observed validation with fixed train-fitted parameters.
            updated = fitted.append(y[train_end:val_end], refit=False)
            tp = updated.forecast(ht)
        return (baseline_positive_prediction(vp, hv), baseline_positive_prediction(tp, ht),
                {"status":"ok", "details":" | ".join(details), "seconds":perf_counter()-started})
    except Exception as exc:
        return (None, None, {"status":"failed", "details":f"{type(exc).__name__}: {exc}",
                            "seconds":perf_counter()-started})

def fit_local_baseline(tag, y, dates, regressors, train_end, val_end, test_end,
                       history_days=0, refit_test=False, order=(1,1,1), maxiter=200):
    """ARIMA retry policy fixed before test. Explicit seasonal fallback if all fits fail."""
    if tag != "ARIMA":
        return _fit_local_baseline_once(tag, y, dates, regressors, train_end, val_end,
                                        test_end, history_days, refit_test, order, maxiter)
    started = time.time()
    # Invalid data must not be concealed by fallback.
    y = np.asarray(y, dtype=np.float64)
    if len(y) != val_end or not np.isfinite(y).all() or (y < 0).any():
        raise ValueError("ARIMA history must be finite, nonnegative and stop at val_end")
    attempts = [(tuple(order), maxiter), (tuple(order), max(500, maxiter)),
                ((1,0,0), 500), ((0,1,1), 500), ((0,1,0), 100)]
    seen, errors = set(), []
    for candidate, iterations in attempts:
        if (candidate, iterations) in seen:
            continue
        seen.add((candidate, iterations))
        v, t, info = _fit_local_baseline_once(
            tag, y, dates, regressors, train_end, val_end, test_end,
            history_days, refit_test, candidate, iterations)
        if info["status"] == "ok":
            info.update(effective_order=str(candidate), attempts=len(seen),
                        used_fallback=False, fallback_method="",
                        retry_errors=" | ".join(errors), seconds=time.time()-started)
            return v, t, info
        errors.append(f"order={candidate}, maxiter={iterations}: {info['details']}")
    def fallback(cutoff, horizon):
        window = min(28, cutoff)
        if window >= 7:
            window = (window//7)*7
            pattern = y[cutoff-window:cutoff].reshape(-1,7).mean(axis=0)
            return np.tile(pattern, int(np.ceil(horizon/7)))[:horizon].astype(np.float32)
        return np.full(horizon, y[:cutoff].mean(), dtype=np.float32)
    return (fallback(train_end, val_end-train_end), fallback(val_end, test_end-val_end),
            dict(status="ok", details="All ARIMA fits failed; explicit seasonal fallback",
                 effective_order="none", attempts=len(seen), used_fallback=True,
                 fallback_method="WEEKDAY_MEAN_4", retry_errors=" | ".join(errors),
                 seconds=time.time()-started))

def run_local_panel(tag):
    vp = np.full((n_series, H), np.nan, np.float32)
    tp = np.full_like(vp, np.nan)
    # Chunk scheduling bounds working memory and persists progress/error logs.
    chunk_size = max(8, 4*max(1, BASELINE_N_JOBS))
    for start in range(0, n_series, chunk_size):
        stop = min(start+chunk_size, n_series)
        results = Parallel(n_jobs=BASELINE_N_JOBS, prefer="threads")(
            delayed(fit_local_baseline)(
                tag, SALES[s, :VAL_END], BASELINE_DATES[:TEST_END],
                BASELINE_REG[s, :TEST_END], TRAIN_END, VAL_END, TEST_END,
                BASELINE_HISTORY_DAYS, BASELINE_REFIT_TEST, ARIMA_ORDER, ARIMA_MAXITER)
            for s in range(start, stop))
        for s, (v, t, status) in zip(range(start, stop), results):
            BASELINE_STATUS.append({"model":tag, "series_id":series_ids[s], **status})
            if v is not None:
                vp[s], tp[s] = v, t
        pd.DataFrame(BASELINE_STATUS).to_csv(BASELINE_DIR / "fit_status.csv", index=False)
        print(f"{tag}: {stop}/{n_series}", flush=True)
    np.savez_compressed(BASELINE_DIR / f"{tag.lower()}_predictions.npz",
                        ids=np.asarray(series_ids), val_pred=vp, test_pred=tp)
    if not (np.isfinite(vp).all() and np.isfinite(tp).all()):
        print(f"{tag}: có fit thất bại; KHÔNG tính metric bằng cách bỏ chuỗi hoặc thay model.")
        return
    output_tag = "ARIMA_ROBUST" if tag == "ARIMA" else tag
    BASELINE_VAL_PRED[output_tag], BASELINE_TEST_PRED[output_tag] = vp, tp
    if tag == "ARIMA":
        statuses = [row for row in BASELINE_STATUS if row["model"] == "ARIMA"]
        print("ARIMA_ROBUST seasonal fallback count:", sum(bool(row.get("used_fallback", False)) for row in statuses))

for _tag in BASELINE_MODELS:
    if _tag in {"ARIMA", "PROPHET"}:
        run_local_panel(_tag)

## 14. Global encoder-decoder LSTM with exogenous inputs
Dùng đúng `train_ds`, `val_ds`, `test_ds`, static IDs và các cửa sổ của TFT.
LSTM encoder đọc 56 ngày, decoder đọc 28 ngày ngoại sinh biết trước; không đưa target vào decoder.
Chọn checkpoint bằng WRMSSE-5 validation (hoặc L12 khi `BEST_METRIC=l12`).
Target/output giữ đơn vị bán gốc; softplus bảo đảm không âm. Không calibration để làm baseline cơ sở.

In [ ]:
class ExogenousLSTMBaseline(nn.Module):
    def __init__(self, n_past, n_future, cards, hidden=64, dropout=0.1):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(card, 8) for card in cards])
        static_dim = 8*len(cards)
        self.encoder = nn.LSTM(n_past+static_dim, hidden, batch_first=True)
        self.decoder = nn.LSTM(n_future+static_dim, hidden, batch_first=True)
        self.output = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, past, future, static):
        context = torch.cat([emb(static[:, j]) for j, emb in enumerate(self.embeddings)], dim=-1)
        enc_input = torch.cat([past, context[:, None].expand(-1, past.size(1), -1)], dim=-1)
        dec_input = torch.cat([future, context[:, None].expand(-1, future.size(1), -1)], dim=-1)
        _, state = self.encoder(enc_input)
        decoded, _ = self.decoder(dec_input, state)
        return F.softplus(self.output(decoded).squeeze(-1))

def train_lstm_baseline():
    seed_everything(SEED)
    net = ExogenousLSTMBaseline(len(MODEL_PAST_COLS), len(FUT_COLS), STATIC_CARD,
                               LSTM_HIDDEN, DROPOUT).to(DEVICE)
    loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=USE_AMP)
    optimizer_lstm = torch.optim.AdamW(net.parameters(), lr=LSTM_LR, weight_decay=WEIGHT_DECAY)
    best, best_weights, wait = float("inf"), None, 0
    records = []
    for epoch in range(1, LSTM_EPOCHS+1):
        started = time.time()
        net.train()
        sum_loss, sum_weight = 0.0, 0.0
        for past, future, static, target, weight, _, _ in loader:
            past, future, static, target, weight = [
                x.to(DEVICE, non_blocking=True) for x in (past, future, static, target, weight)]
            optimizer_lstm.zero_grad(set_to_none=True)
            forecast = net(past, future, static)
            loss = (((forecast-target)**2)*weight).sum()/weight.sum().clamp_min(1)
            if not torch.isfinite(loss):
                raise FloatingPointError("LSTM loss NaN/Inf; không xuất kết quả không hợp lệ")
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer_lstm.step()
            sum_loss += loss.item()*weight.sum().item()
            sum_weight += weight.sum().item()
        vpred, truth, _ = predict_ds(net, val_ds)
        assert np.allclose(truth, BASELINE_VAL_TRUE)
        frame, wr = eval_all(truth, vpred, VAL_EVAL_CONTEXTS, "LSTM")
        score = float(frame.loc[frame.level.eq("L12"), "WRMSSE"].iloc[0]) if BEST_METRIC=="l12" else wr
        if not np.isfinite(score):
            raise FloatingPointError("LSTM validation score không hữu hạn")
        records.append({"epoch":epoch, "train_weighted_mse":sum_loss/max(sum_weight,1),
                        "val_wrmsse5":wr, "selection_score":score, "seconds":time.time()-started})
        pd.DataFrame(records).to_csv(BASELINE_DIR / "lstm_history.csv", index=False)
        print(records[-1], flush=True)
        if score < best-1e-4:
            best, wait = score, 0
            best_weights = {k:v.detach().cpu().clone() for k,v in net.state_dict().items()}
        else:
            wait += 1
        if wait >= LSTM_PATIENCE:
            break
    if best_weights is None:
        raise RuntimeError("Không có LSTM checkpoint hợp lệ")
    net.load_state_dict(best_weights)
    vp, _, _ = predict_ds(net, val_ds)
    tp, truth, _ = predict_ds(net, test_ds)
    assert np.allclose(truth, BASELINE_TEST_TRUE)
    if not (np.isfinite(vp).all() and np.isfinite(tp).all()):
        raise FloatingPointError("LSTM prediction NaN/Inf")
    # Store preprocessing and mappings needed to reproduce the model's input coding.
    mappings = {col:sorted(STATIC_DF[col].astype(str).unique().tolist()) for col in STATIC_COLS}
    torch.save({"state_dict":best_weights, "hidden":LSTM_HIDDEN, "dropout":DROPOUT,
                "static_card":STATIC_CARD, "static_categories_codes_start_at_1":mappings,
                "past_cols":MODEL_PAST_COLS, "future_cols":FUT_COLS,
                "feature_cols":FEATURE_COLS, "feat_mean":FEAT_MEAN, "feat_std":FEAT_STD,
                "local_mean":LOCAL_STATE_MEAN, "local_std":LOCAL_STATE_STD,
                "hier_mean":HIER_CONTEXT_MEAN, "hier_std":HIER_CONTEXT_STD,
                "seq":SEQ, "horizon":H, "best_score":best,
                "train_end":TRAIN_END, "loss":"weighted_mse", "seed":SEED},
               BASELINE_DIR / "lstm_baseline.pt")
    np.savez_compressed(BASELINE_DIR / "lstm_predictions.npz", ids=np.asarray(series_ids),
                        val_pred=vp, test_pred=tp)
    return vp, tp

if "LSTM" in BASELINE_MODELS:
    BASELINE_VAL_PRED["LSTM"], BASELINE_TEST_PRED["LSTM"] = train_lstm_baseline()

## 15. Compare every model on the SAME full validation/test horizons
Mọi metric được tính sau khi tổng hợp bottom-up, không lấy trung bình metric SKU để thay cho store metric.
Đưa TFT_BU và biến thể TFT đã chọn trên validation vào bảng test. Không chọn lại model bằng test.
Validation của TFT/LSTM đã dùng để chọn checkpoint, nên không phải ước lượng độc lập cuối cùng.
TFT calibrated chỉ báo cáo test; không chấm calibration trên chính validation đã fit hệ số.

### L12: hiệu chỉnh co về BU, chọn trên validation
Thử alpha = 0/0.25/0.5/0.75/1 và k = 5/20/100. Chọn L12 WRMSSE nhỏ nhất với L3 không xấu hơn BU quá 2% trên 14 ngày validation cuối. Alpha=0 giữ nguyên BU. Sau chọn mới fit lại factors bằng toàn validation. Đây là mục tiêu lựa chọn mới, khác SERIES_CAL chọn theo WRMSSE-5; báo cả hai. Không chấm dự báo đã fit factors trên chính full validation. Checkpoint đã dùng full validation, nên selection tail chưa hoàn toàn độc lập; cần thêm rolling backtest để xác nhận. Không bảo đảm cải thiện trên test.


In [ ]:
# L12 calibration: select on held-out validation tail, never on test.
# Include alpha=0 so the original BU forecast is a valid candidate.
L12_STORE_TOLERANCE = float(os.getenv("L12_STORE_TOLERANCE", "0.02"))
assert L12_STORE_TOLERANCE >= 0
cal_h = CALIBRATION_DAYS
base_select, _ = eval_all(val_true[:, cal_h:], val_pred[:, cal_h:], CAL_SELECT_EVAL_CONTEXTS, "BU")
base_store = float(base_select.loc[base_select.level.eq("L3"), "WRMSSE"].iloc[0])
rows = []
for k in [5.0, 20.0, 100.0]:
    factors = fit_series_factors(val_pred[:, :cal_h], val_true[:, :cal_h], k=k)
    for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
        pred = val_pred[:, cal_h:] * (1 + alpha*(factors-1))[:, None]
        frame, overall = eval_all(val_true[:, cal_h:], pred, CAL_SELECT_EVAL_CONTEXTS, "candidate")
        sku = float(frame.loc[frame.level.eq("L12"), "WRMSSE"].iloc[0])
        store = float(frame.loc[frame.level.eq("L3"), "WRMSSE"].iloc[0])
        rows.append(dict(k=k, alpha=alpha, L12_WRMSSE=sku, L3_WRMSSE=store,
                         WRMSSE_5levels=overall, eligible=store <= base_store*(1+L12_STORE_TOLERANCE)+1e-12))
l12_selection = pd.DataFrame(rows)
l12_choice = l12_selection[l12_selection.eligible].sort_values(["L12_WRMSSE", "alpha", "k"]).iloc[0]
# Refit factors using all observed validation only after choosing alpha and k.
l12_factors = fit_series_factors(val_pred, val_true, k=float(l12_choice.k))
l12_factors = 1 + float(l12_choice.alpha)*(l12_factors-1)
l12_test = test_variants["BU"] * l12_factors[:, None]
BASELINE_TEST_PRED["TFT_L12_SHRINK"] = l12_test
l12_selection.to_csv(BASELINE_DIR / "l12_calibration_selection.csv", index=False)
np.savez_compressed(BASELINE_DIR / "l12_shrink_predictions.npz", ids=np.asarray(series_ids),
                    test_pred=l12_test, factors=l12_factors)
with open(BASELINE_DIR / "l12_calibration_config.json", "w") as f:
    json.dump(dict(alpha=float(l12_choice.alpha), k=float(l12_choice.k),
                   store_tolerance=L12_STORE_TOLERANCE,
                   selection_days=[VAL_START+cal_h, VAL_END],
                   selection_metric="L12_WRMSSE", strategy="shrink series factors directly from BU"), f, indent=2)
print("L12 calibration chosen only on validation:", l12_choice.to_dict())
print("alpha=0 means BU was retained. Test improvement is NOT guaranteed.")


In [ ]:
for _tag, _builder in [
    ("NAIVE_7", lambda start: seasonal_naive(SALES,start,7,H)),
    ("NAIVE_28", lambda start: seasonal_naive(SALES,start,28,H)),
    ("WEEKDAY_MEAN_4", lambda start: weekday_mean_naive(SALES,start,H,4))]:
    BASELINE_VAL_PRED[_tag] = _builder(VAL_START)
    BASELINE_TEST_PRED[_tag] = _builder(TEST_START)
BASELINE_VAL_PRED["TFT_BU"] = val_pred.copy()
BASELINE_TEST_PRED["TFT_BU"] = test_variants["BU"].copy()
if best_tag != "BU":
    BASELINE_TEST_PRED["TFT_SELECTED_"+best_tag] = test_variants[best_tag].copy()

comparison_frames = []
for split, truth, predictions, contexts in [
    ("validation", BASELINE_VAL_TRUE, BASELINE_VAL_PRED, VAL_EVAL_CONTEXTS),
    ("test", BASELINE_TEST_TRUE, BASELINE_TEST_PRED, TEST_EVAL_CONTEXTS)]:
    for tag, pred in predictions.items():
        if pred.shape != truth.shape or not np.isfinite(pred).all():
            raise ValueError(f"{tag}/{split}: shape hoặc giá trị không hợp lệ")
        frame, _ = eval_all(truth, pred, contexts, tag)
        frame.insert(0, "split", split)
        frame["MAPE_nonzero_pct"] = 100*frame["MAPE"]
        frame["WMAPE_pct"] = 100*frame["WMAPE"]
        frame["aggregation_scope"] = "full_M5" if n_series == 30490 else "selected_series_only"
        comparison_frames.append(frame)
comparison_df = pd.concat(comparison_frames, ignore_index=True)
comparison_df.to_csv(BASELINE_DIR / "comparison_all_levels.csv", index=False)
main_table = comparison_df[comparison_df.level.isin(["L12", "L3"])][
    ["split","model","level","n_keys","RMSE","MAE","MAPE_nonzero_pct","WMAPE_pct","WRMSSE","Bias","aggregation_scope"]]
main_table.to_csv(BASELINE_DIR / "comparison_sku_store.csv", index=False)
print(main_table.to_string(index=False))
summary_baselines = comparison_df.groupby(["split","model"], sort=False)["WRMSSE"].mean().reset_index(name="WRMSSE_5levels")
summary_baselines.to_csv(BASELINE_DIR / "comparison_summary.csv", index=False)
print("\nModel coverage:", {tag:("ARIMA_ROBUST" if tag == "ARIMA" else tag) in BASELINE_TEST_PRED for tag in BASELINE_MODELS})
print("Nếu model thiếu khỏi bảng, xem fit_status.csv; không công bố bảng như đã so sánh đủ baseline.")

## 16. Per-store errors and experiment manifest
Metric từng store giúp nhận biết lỗi bị che khuất khi chỉ nhìn tổng. Manifest lưu cấu hình thực chạy,
phiên bản thư viện, phạm vi chuỗi và protocol cập nhật trước test.

In [ ]:
store_error_rows = []
for tag, pred in BASELINE_TEST_PRED.items():
    for group in np.unique(GROUPS["L3"]):
        members = GROUPS["L3"] == group
        yt, yp = BASELINE_TEST_TRUE[members].sum(axis=0), pred[members].sum(axis=0)
        err, positive = yp-yt, yt>0
        denom = float(np.abs(yt).sum())
        store_error_rows.append({"model":tag,
            "store_id":str(sales_sel.loc[members,"store_id"].iloc[0]),
            "n_items":int(members.sum()), "RMSE":float(np.sqrt(np.mean(err**2))),
            "MAE":float(np.abs(err).mean()),
            "MAPE_nonzero_pct":float(100*np.mean(np.abs(err[positive])/yt[positive])) if positive.any() else np.nan,
            "WMAPE_pct":float(100*np.abs(err).sum()/denom) if denom>0 else np.nan,
            "Bias":float(err.sum()/denom) if denom>0 else np.nan})
pd.DataFrame(store_error_rows).to_csv(BASELINE_DIR / "test_per_store.csv", index=False)
versions = {}
for package in ["numpy","pandas","torch","statsmodels","prophet","joblib"]:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None
manifest = {
    "requested_models":BASELINE_MODELS,
    "completed_models":[x for x in BASELINE_MODELS if ("ARIMA_ROBUST" if x == "ARIMA" else x) in BASELINE_TEST_PRED],
    "n_series":n_series, "profile":PROFILE, "seed":SEED,
    "train_end":TRAIN_END, "val_start":VAL_START,"val_end":VAL_END,
    "test_start":TEST_START,"test_end":TEST_END,"horizon":H,"seq":SEQ,
    "statistical_history_days":BASELINE_HISTORY_DAYS,
    "statistical_refit_before_test":BASELINE_REFIT_TEST,
    "arima_order":ARIMA_ORDER,"arima_maxiter":ARIMA_MAXITER,
    "prophet_regressors":PROPHET_REGRESSORS,
    "lstm_epochs_max":LSTM_EPOCHS,"lstm_hidden":LSTM_HIDDEN,"lstm_lr":LSTM_LR,
    "lstm_patience":LSTM_PATIENCE,"lstm_loss":"weighted_mse",
    "lstm_hierarchical_loss":False,"lstm_occurrence_head":False,
    "series_weight_power":SERIES_WEIGHT_POWER,"train_origin_count":len(train_origin_values),
    "selection_metric":BEST_METRIC,"tft_selected_on_validation":best_tag,
    "mape_policy":"actual > 0 only; percentages in *_pct columns",
    "aggregation_scope":"full_M5" if n_series==30490 else "selected_series_only",
    "versions":versions,
}
with open(BASELINE_DIR / "baseline_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
pd.DataFrame({"series_id":series_ids}).to_csv(BASELINE_DIR / "series_order.csv",index=False)
print("Baseline outputs:", BASELINE_DIR.resolve())

## 17. Đọc kết quả và giới hạn so sánh
- `comparison_sku_store.csv`: bảng chính theo đề cương, full 28 ngày, SKU–store (L12) và store (L3).
- `comparison_all_levels.csv`: RMSE, MAE, MAPE, WMAPE, WRMSSE, Bias tại 5 cấp.
- `comparison_summary.csv`: WRMSSE-5 trung bình theo model và split; không dùng test để chọn model.
- `test_per_store.csv`: chi tiết từng cửa hàng.
- `fit_status.csv`: tình trạng ARIMA/Prophet; các trường hợp hằng, cảnh báo hội tụ và lỗi được ghi rõ.
- `*_predictions.npz`: dự báo theo đúng `ids`; NaN chỉ xuất hiện ở model thống kê bị lỗi và model đó không được chấm điểm hoàn chỉnh.
- `lstm_history.csv`, `lstm_baseline.pt`: lịch sử huấn luyện và checkpoint tốt nhất.
- `baseline_manifest.json`: protocol và phiên bản thực tế; cần đính kèm khi báo cáo.

ARIMA mặc định là fixed-order, không auto-tuning. Prophet/LSTM cũng chưa được tìm siêu tham số.
Tuning chỉ thực hiện trên validation hoặc rolling validation, không dùng test.
ARIMA là univariate; Prophet dùng ngoại sinh lịch; LSTM/TFT dùng bộ feature giàu hơn. Muốn cô lập tác động attention,
cần thêm ablation dùng cùng feature, cùng loss và ngân sách tuning, không chỉ so sánh tên kiến trúc.

Validation/test là backtest nội bộ, không phải submission dự báo phần hidden của cuộc thi Kaggle.
Không diễn giải zero sales thành missing values. Dự báo âm của mô hình thống kê được chặn về 0, áp dụng nhất quán ở cả val/test.
Bộ evaluator WRMSSE-5 gốc bỏ các chuỗi scale không hợp lệ; xem `n_valid_scales` khi diễn giải kết quả.